# Loans at Risk: Capturing Default — Extract · Transform · Load

## Purpose

This notebook implements the ETL layer of the LendingClub default prediction project.  
The training window covers **2007–2014**; the test window covers **2015**.

The objective is not analysis. It is structural control.

Raw CSV extracts are converted into datasets that are suitable for downstream modeling under a clearly defined constraint: prediction is made at the moment of application submission.

“Structurally reliable” in this context means:

- Identical schema across train and test  
- Explicit treatment of datatypes and null structure  
- Removal of null-only and constant columns  
- Elimination of export artifacts  
- Early enforcement of the submission-time prediction boundary  

No interpretation is performed here. This notebook establishes a clean, governed input layer so that later analytical steps operate on stable ground.

All dependency management is handled at repository level (`README`, `requirements.txt`). This notebook assumes the environment is already configured.

---

## Extract

Raw LendingClub loan data is ingested directly from the source CSV files.

The two temporal partitions are kept strictly separate:

- **2007–2014** → training  
- **2015** → testing  

Extraction preserves the raw state of the data. No filtering, type coercion, or feature selection occurs at this stage. The raw layer is treated as immutable input.

---

## Transform

The transformation phase enforces structural and temporal discipline.

Concretely:

- Schema and datatype validation  
- Identification and removal of null-only and constant columns  
- Detection of mixed types and object-encoded numerics  
- Removal of structural artifacts (e.g., exported index columns)  
- Explicit exclusion of post-origination, underwriting, and pricing variables  
- Deterministic type conversions (including numeric and date normalization)  
- Structured handling of credit-event timing variables, where null represents absence rather than data loss  
- Identical transformation logic across training and test sets  

No feature engineering beyond structural normalization is performed.  
No modeling assumptions are introduced.

The output of this phase is a submission-time compliant dataset.

---

## Load

The transformed datasets are persisted in Parquet format.

Two layers are materialized:

- **Clean dataset** → structurally aligned variables  
- **Feature-base dataset** → submission-time eligible variables plus target  

Schema identity between training and test is validated prior to persistence.

These outputs form the controlled input layer for EDA and modeling.

---

## Scope Boundary

Feature engineering, modeling, evaluation, and decision logic are handled in subsequent notebooks.

This notebook exists to ensure that those stages operate on data that is structurally sound, temporally consistent, and reproducible.

In [1]:
# -------------------------------
# Imports: libraries and custom functions
# -------------------------------

from datetime import datetime, timezone
from pathlib import Path
from typing import Callable
import uuid

import pandas as pd

# Editable-installed local package
import python
from config.logging import (
    get_logger,
    log_category_differences,
    log_messages,
)
from dataset.preprocessing import (
    apply_categorical_mapping,
    apply_credit_sentinel_encoding,
    audit_string_columns,
    compare_categorical_column_values,
    convert_month_year_columns_to_datetime,
    create_row_identifier,
    drop_columns_with_logging,
    initial_inspection,
    normalize_home_ownership,
    normalize_text_columns,
    transform_emp_length,
    build_combined_schema,
    build_structural_issues_report,
    build_string_alignment_report
)

In [2]:
# -------------------------------
# Project root
# -------------------------------

package_init_path = Path(python.__file__).resolve()
project_root = package_init_path.parents[2]  # .../src/python/__init__.py -> repo root

if not (project_root / "src").exists():
    raise RuntimeError(
        f"Failed to resolve project_root. Expected 'src' directory at: {project_root}"
    )

In [3]:
# ===============================
# Paths and run context
# ===============================

logs_root = project_root / "logs"
logs_root.mkdir(parents=True, exist_ok=True)

PROJECT_LOG_FILE = logs_root / "project.log"

run_id = uuid.uuid4().hex[:10]
run_timestamp_utc = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
run_header = f"NEW RUN: {run_timestamp_utc} UTC | RUN_ID={run_id}"

log_messages("\n" + "=" * 60, PROJECT_LOG_FILE)
log_messages(run_header, PROJECT_LOG_FILE)
log_messages("=" * 60 + "\n", PROJECT_LOG_FILE)

print(run_header)

log: Callable[[str], None] = get_logger(PROJECT_LOG_FILE)
log("ETL notebook initialized")
log(f"project_root: {project_root}")


# ---------------------------------------------------------------
# Inputs for this notebook (raw, interim, report)
# ---------------------------------------------------------------

raw_training_data_file = project_root / "data" / "external" / "loan_data_2007_2014.csv"
raw_test_data_file = project_root / "data" / "external" / "loan_data_2015.csv"

clean_training_data_file = project_root / "data" / "interim" / "clean_loan_data_2007_2014.parquet"
clean_test_data_file = project_root / "data" / "interim" / "clean_loan_data_2015.parquet"

feature_base_training_data_file = project_root / "data" / "interim" / "feature_base_loan_data_2007_2014.parquet"
feature_base_test_data_file = project_root / "data" / "interim" / "feature_base_loan_data_2015.parquet"

audit_dir = project_root / "artifacts" / "audit"

appendix_base_name = "appendix_A_raw_feature_universe"

appendix_parquet_file = audit_dir / f"{appendix_base_name}.parquet"
appendix_csv_file = audit_dir / f"{appendix_base_name}.csv"

required_directories = {
    raw_training_data_file.parent,
    raw_test_data_file.parent,
    clean_training_data_file.parent,
    clean_test_data_file.parent,
    feature_base_training_data_file.parent,
    feature_base_test_data_file.parent,
    appendix_parquet_file.parent,
    appendix_csv_file.parent,
}

for directory_path in sorted(required_directories):
    directory_path.mkdir(parents=True, exist_ok=True)

log(f"Raw training data path: {raw_training_data_file}")
log(f"Raw test data path: {raw_test_data_file}")
log(f"Clean training parquet path: {clean_training_data_file}")
log(f"Clean test parquet path: {clean_test_data_file}")
log(f"Feature base training parquet path: {feature_base_training_data_file}")
log(f"Feature base test parquet path: {feature_base_test_data_file}")
log(f"Appendix parquet path: {appendix_parquet_file}")
log(f"Appendix CSV path: {appendix_csv_file}")

NEW RUN: 2026-02-28 16:11:45 UTC | RUN_ID=1ccc8e6024


## Initial Data Inspection

This step examines the raw dataset exactly as delivered.

The objective is to map its structural characteristics before any transformation rules are applied.

The inspection evaluates:

- Column count and schema alignment  
- Datatype assignments and object-encoded numerics  
- Fully null and constant columns  
- Patterns of missingness, including variables where null reflects structural absence  
- Mixed-type inconsistencies  
- Identifier fields and exported artifacts  

No variables are removed and no values are altered at this stage.

Findings from this inspection determine which columns require exclusion, normalization, or explicit encoding during transformation. Every later modification is grounded in observations documented here.

In [4]:
# --------------------------------------------------------
# Load raw datasets (train/test) + checkpoint
# --------------------------------------------------------

df_raw_training_data = pd.read_csv(raw_training_data_file, low_memory=False)
df_raw_test_data = pd.read_csv(raw_test_data_file, low_memory=False)

raw_overview_df = pd.DataFrame(
    [
        {
            "dataset": "train_raw",
            "rows": df_raw_training_data.shape[0],
            "cols": df_raw_training_data.shape[1],
            "memory_mb": round(df_raw_training_data.memory_usage(deep=True).sum() / (1024**2), 2),
        },
        {
            "dataset": "test_raw",
            "rows": df_raw_test_data.shape[0],
            "cols": df_raw_test_data.shape[1],
            "memory_mb": round(df_raw_test_data.memory_usage(deep=True).sum() / (1024**2), 2),
        },
    ]
)

display(raw_overview_df)
log(f"[raw] overview: {raw_overview_df.to_dict(orient='records')}")

,dataset,rows,cols,memory_mb
0,train_raw,466285,75,389.20
1,test_raw,421094,74,322.85


In [5]:
# -----------------------------------------
# Raw schema comparison (train vs test)
# -----------------------------------------

combined_schema_raw = build_combined_schema(
    train_dataframe=df_raw_training_data,
    test_dataframe=df_raw_test_data,
    stage_name="raw",
    log=log,
)

# Small summary (high signal)
schema_summary_df = pd.DataFrame([{
    "train_rows": df_raw_training_data.shape[0],
    "test_rows": df_raw_test_data.shape[0],
    "train_cols": df_raw_training_data.shape[1],
    "test_cols": df_raw_test_data.shape[1],
    "train_only_count": int((combined_schema_raw["present_in_train"] & ~combined_schema_raw["present_in_test"]).sum()),
    "test_only_count": int((~combined_schema_raw["present_in_train"] & combined_schema_raw["present_in_test"]).sum()),
    "dtype_mismatch_count": int(
        (combined_schema_raw["present_in_train"]
         & combined_schema_raw["present_in_test"]
         & combined_schema_raw["dtype_mismatch"]).sum()
    ),
}])

display(schema_summary_df)
log(f"[raw][schema] notebook_summary: {schema_summary_df.to_dict(orient='records')[0]}")

# Delta-only schema view (columns present in only one split OR dtype mismatch)
schema_deltas_df = combined_schema_raw.loc[
    (combined_schema_raw["present_in_train"] ^ combined_schema_raw["present_in_test"])
    | (
        combined_schema_raw["present_in_train"]
        & combined_schema_raw["present_in_test"]
        & combined_schema_raw["dtype_mismatch"]
    ),
    ["column_name", "train_dtype", "test_dtype", "present_in_train", "present_in_test", "dtype_mismatch"],
].copy()

schema_deltas_df = (
    schema_deltas_df
    .sort_values(["dtype_mismatch", "column_name"], ascending=[False, True])
    .reset_index(drop=True)
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(schema_deltas_df)

log(f"[raw][schema] delta_rows={schema_deltas_df.shape[0]}")

,train_rows,test_rows,train_cols,test_cols,train_only_count,test_only_count,dtype_mismatch_count
0,466285,421094,75,74,1,0,1


,column_name,train_dtype,test_dtype,present_in_train,present_in_test,dtype_mismatch
0,verification_status_joint,float64,str,True,True,True
1,Unnamed: 0,int64,NaN,True,False,False


In [6]:
# Initial inspection of the raw training and testing data
inspection_summary_train = initial_inspection(df_raw_training_data, PROJECT_LOG_FILE)
inspection_summary_test = initial_inspection(df_raw_test_data, PROJECT_LOG_FILE)

In [7]:
# ------------------------------------------------------------------
# Feature universe overview (raw train vs raw test)
# For Appendix A
# ------------------------------------------------------------------

inspection_summary_train = initial_inspection(df_raw_training_data, PROJECT_LOG_FILE)
inspection_summary_test = initial_inspection(df_raw_test_data, PROJECT_LOG_FILE)

feature_summary_train_df = inspection_summary_train["feature_summary"].copy()
feature_summary_test_df = inspection_summary_test["feature_summary"].copy()

train_overview_df = (
    feature_summary_train_df
    .reset_index()
    .rename(columns={
        "index": "column_name",
        "dtype": "train_dtype_inferred",
        "n_unique": "train_n_unique",
        "null_percent": "train_null_percent",
        "is_numeric": "train_is_numeric",
        "is_object": "train_is_object",
        "is_mixed_type": "train_is_mixed_type",
        "is_numeric_object": "train_is_numeric_object",
        "is_fully_null": "train_is_fully_null",
        "is_constant": "train_is_constant",
    })
)

test_overview_df = (
    feature_summary_test_df
    .reset_index()
    .rename(columns={
        "index": "column_name",
        "dtype": "test_dtype_inferred",
        "n_unique": "test_n_unique",
        "null_percent": "test_null_percent",
        "is_numeric": "test_is_numeric",
        "is_object": "test_is_object",
        "is_mixed_type": "test_is_mixed_type",
        "is_numeric_object": "test_is_numeric_object",
        "is_fully_null": "test_is_fully_null",
        "is_constant": "test_is_constant",
    })
)

combined_feature_universe_df = pd.merge(
    train_overview_df,
    test_overview_df,
    on="column_name",
    how="outer",
)

combined_feature_universe_df["max_null_percent"] = combined_feature_universe_df[
    ["train_null_percent", "test_null_percent"]
].max(axis=1)

combined_feature_universe_df["null_gap_test_minus_train"] = (
    combined_feature_universe_df["test_null_percent"] - combined_feature_universe_df["train_null_percent"]
)

combined_feature_universe_df = pd.merge(
    combined_feature_universe_df,
    combined_schema_raw[["column_name", "present_in_train", "present_in_test", "train_dtype", "test_dtype", "dtype_mismatch"]],
    on="column_name",
    how="left",
)

# Severity sorting: mismatches + missingness + structural red flags first
combined_feature_universe_df["severity"] = 0
combined_feature_universe_df.loc[combined_feature_universe_df["dtype_mismatch"] == True, "severity"] += 100
combined_feature_universe_df.loc[
    (combined_feature_universe_df["present_in_train"] ^ combined_feature_universe_df["present_in_test"]) == True,
    "severity"
] += 80
combined_feature_universe_df.loc[
    (combined_feature_universe_df["train_is_fully_null"] == True) | (combined_feature_universe_df["test_is_fully_null"] == True),
    "severity"
] += 60
combined_feature_universe_df.loc[
    (combined_feature_universe_df["train_is_constant"] == True) | (combined_feature_universe_df["test_is_constant"] == True),
    "severity"
] += 40
combined_feature_universe_df.loc[combined_feature_universe_df["max_null_percent"] >= 50, "severity"] += 10

combined_feature_universe_df = (
    combined_feature_universe_df
    .sort_values(["severity", "max_null_percent", "column_name"], ascending=[False, False, True])
    .drop(columns=["severity"])
    .reset_index(drop=True)
)

log(f"[raw][universe] rows={combined_feature_universe_df.shape[0]} cols={combined_feature_universe_df.shape[1]}")

In [8]:
#-------------------------------------------------------------------------
# Actionable insights: columns with dtype mismatches, 
# presence mismatches, high nulls, or constant values
#-------------------------------------------------------------------------

action_df = combined_feature_universe_df.loc[
    (combined_feature_universe_df["dtype_mismatch"] == True)
    | ((combined_feature_universe_df["present_in_train"] ^ combined_feature_universe_df["present_in_test"]) == True)
    | (combined_feature_universe_df["max_null_percent"] >= 50)
    | (combined_feature_universe_df["train_is_constant"] == True)
    | (combined_feature_universe_df["test_is_constant"] == True),
    [
        "column_name",
        "present_in_train",
        "present_in_test",
        "train_dtype",
        "test_dtype",
        "dtype_mismatch",
        "train_null_percent",
        "test_null_percent",
        "max_null_percent",
        "null_gap_test_minus_train",
        "train_is_fully_null",
        "test_is_fully_null",
        "train_is_constant",
        "test_is_constant",
    ],
].copy()

action_df = action_df.sort_values(
    ["dtype_mismatch", "max_null_percent", "column_name"],
    ascending=[False, False, True],
).reset_index(drop=True)

display(action_df)
log(f"[raw][universe] action_rows={action_df.shape[0]}")

,column_name,present_in_train,present_in_test,train_dtype,test_dtype,dtype_mismatch,train_null_percent,test_null_percent,max_null_percent,null_gap_test_minus_train,train_is_fully_null,test_is_fully_null,train_is_constant,test_is_constant
0,verification_status_joint,True,True,float64,str,True,100.00,99.88,100.00,-0.12,True,False,False,False
1,all_util,True,True,float64,float64,False,100.00,94.92,100.00,-5.08,True,False,False,False
2,annual_inc_joint,True,True,float64,float64,False,100.00,99.88,100.00,-0.12,True,False,False,False
3,dti_joint,True,True,float64,float64,False,100.00,99.88,100.00,-0.12,True,False,False,False
4,il_util,True,True,float64,float64,False,100.00,95.58,100.00,-4.42,True,False,False,False
5,inq_fi,True,True,float64,float64,False,100.00,94.92,100.00,-5.08,True,False,False,False
6,inq_last_12m,True,True,float64,float64,False,100.00,94.92,100.00,-5.08,True,False,False,False
7,max_bal_bc,True,True,float64,float64,False,100.00,94.92,100.00,-5.08,True,False,False,False
8,mths_since_rcnt_il,True,True,float64,float64,False,100.00,95.06,100.00,-4.94,True,False,False,False
9,open_acc_6m,True,True,float64,float64,False,100.00,94.92,100.00,-5.08,True,False,False,False


In [9]:
# -----------------------------------------
# Appendix export: Raw feature universe
# -----------------------------------------

combined_feature_universe_export_df = combined_feature_universe_df.drop(
    columns=["train_dtype_inferred", "test_dtype_inferred"],
    errors="ignore",
).copy()

combined_feature_universe_export_df["train_dtype"] = combined_feature_universe_export_df["train_dtype"].astype("string")
combined_feature_universe_export_df["test_dtype"] = combined_feature_universe_export_df["test_dtype"].astype("string")

combined_feature_universe_export_df.to_parquet(appendix_parquet_file, index=False)
combined_feature_universe_export_df.to_csv(appendix_csv_file, index=False, encoding="utf-8")

print(f"Appendix A saved to: {appendix_csv_file}")
print(f"Appendix A saved to: {appendix_parquet_file}")
log(f"[raw][appendix] parquet_saved={appendix_parquet_file}")
log(f"[raw][appendix] csv_saved={appendix_csv_file}")

Appendix A saved to: D:\Portfolio\loans-at-risk-capturing-default\artifacts\audit\appendix_A_raw_feature_universe.csv
Appendix A saved to: D:\Portfolio\loans-at-risk-capturing-default\artifacts\audit\appendix_A_raw_feature_universe.parquet


In [10]:
# Creating a structural issues report for the raw feature universe
structural_issues_df = build_structural_issues_report(
    combined_feature_universe_df=combined_feature_universe_df,
    log=log,
)

In [11]:
# Structural issues summary (high signal)
issue_counts_df = (
    structural_issues_df
    .groupby(["issue"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values(["count", "issue"], ascending=[False, True])
    .reset_index(drop=True)
)

display(issue_counts_df)
log(f"[raw][structural_issues] issue_counts={issue_counts_df.to_dict(orient='records')}")

,issue,count
0,High Missing (>50%),24
1,100% Null,17
2,Constant,3
3,Present in train only,1
4,Dtype mismatch,1


In [12]:
issue_priority = [
    "Present in train only",
    "Present in test only",
    "Dtype mismatch",
    "100% Null",
    "High Missing (>50%)",
    "Constant",
]

examples_df = (
    structural_issues_df
    .assign(issue=pd.Categorical(structural_issues_df["issue"], categories=issue_priority, ordered=True))
    .sort_values(["issue", "column_name", "applies_to"])
    .groupby("issue", as_index=False)
    .head(5)
    .reset_index(drop=True)
)

display(examples_df)
log(f"[raw][structural_issues] example_rows={examples_df.shape[0]}")

,column_name,issue,applies_to
0,Unnamed: 0,Present in train only,train
1,verification_status_joint,Dtype mismatch,both
2,all_util,100% Null,train
3,annual_inc_joint,100% Null,train
4,dti_joint,100% Null,train
5,il_util,100% Null,train
6,inq_fi,100% Null,train
7,all_util,High Missing (>50%),test
8,annual_inc_joint,High Missing (>50%),test
9,desc,High Missing (>50%),test


In [13]:
action_map = {
    "Present in train only": "Drop ingestion artifact / align schema",
    "Present in test only": "Align schema (investigate source) / drop if absent in train",
    "Dtype mismatch": "Coerce to consistent dtype (string/category/numeric)",
    "100% Null": "Drop (non-informative in this cohort)",
    "High Missing (>50%)": "Flag as regime-shift sensitive; likely drop or isolate",
    "Constant": "Drop (non-informative)",
}

decision_queue_df = (
    structural_issues_df
    .assign(recommended_action=structural_issues_df["issue"].map(action_map))
    .drop_duplicates(subset=["column_name", "issue", "applies_to"])
    .sort_values(["issue", "column_name", "applies_to"])
    .reset_index(drop=True)
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(decision_queue_df.head(25))
log(f"[raw][structural_issues] decision_queue_rows={decision_queue_df.shape[0]}")

,column_name,issue,applies_to,recommended_action
0,Unnamed: 0,Present in train only,train,Drop ingestion artifact / align schema
1,verification_status_joint,Dtype mismatch,both,Coerce to consistent dtype (string/category/numeric)
2,all_util,100% Null,train,Drop (non-informative in this cohort)
3,annual_inc_joint,100% Null,train,Drop (non-informative in this cohort)
4,dti_joint,100% Null,train,Drop (non-informative in this cohort)
5,il_util,100% Null,train,Drop (non-informative in this cohort)
6,inq_fi,100% Null,train,Drop (non-informative in this cohort)
7,inq_last_12m,100% Null,train,Drop (non-informative in this cohort)
8,max_bal_bc,100% Null,train,Drop (non-informative in this cohort)
9,mths_since_rcnt_il,100% Null,train,Drop (non-informative in this cohort)


In [14]:
reports = build_string_alignment_report(
	training_dataframe=df_raw_training_data,
	test_dataframe=df_raw_test_data,
	sample_size=5,
	top_k=15,
	drilldown_max_columns=5,
	drilldown_top_values_per_side=15,
	log_file=PROJECT_LOG_FILE,
	export_dir=None,
)

reports["summary"]

,string_like_cols_train,string_like_cols_test,string_like_cols_union,string_like_cols_with_differences,top_k_returned
0,22,23,23,16,15


In [17]:
reports["top_deltas"]

,column_name,dtype_train,unique_including_null_train,unique_non_null_train,null_percent_train,sample_values_train,dtype_test,unique_including_null_test,unique_non_null_test,null_percent_test,sample_values_test,present_in_train,present_in_test,dtype_mismatch,unique_non_null_gap_test_minus_train,null_gap_test_minus_train,max_null_percent,has_difference
0,verification_status_joint,NaN,NaN,NaN,NaN,NaN,str,4,3,99.88,"[Not Verified, Verified, Source Verified]",False,True,False,3.0,NaN,99.88,True
1,next_pymnt_d,str,101.0,100.0,48.73,"[Feb-16, Jan-16, Sep-13, Feb-14, May-14]",str,4,3,6.12,"[Jan-16, Feb-16, Mar-16]",True,True,False,-97.0,-42.61,48.73,True
2,desc,str,124436.0,124435.0,72.98,[Borrower added on 12/22/11 > I need to upgrad...,str,35,34,99.99,[We knew that using our credit cards to financ...,True,True,False,-124401.0,27.01,99.99,True
3,last_pymnt_d,str,99.0,98.0,0.08,"[Jan-15, Apr-13, Jun-14, Jan-16, Apr-12]",str,14,13,4.10,"[Jan-16, Dec-15, Nov-15, Oct-15, Sep-15]",True,True,False,-85.0,4.02,4.10,True
4,emp_title,str,205476.0,205475.0,5.92,"[Ryder, AIR RESOURCES BOARD, University Medica...",str,120812,120811,5.67,"[Foreign Service Officer, Associate Consultant...",True,True,False,-84664.0,-0.25,5.92,True
5,title,str,63099.0,63098.0,0.00,"[Computer, bike, real estate business, persone...",str,28,27,0.03,"[Home improvement, Credit card refinancing, De...",True,True,False,-63071.0,0.03,0.03,True
6,last_credit_pull_d,str,104.0,103.0,0.01,"[Jan-16, Sep-13, Jan-15, Sep-15, Dec-14]",str,15,14,0.00,"[Jan-16, Dec-15, Nov-15, Sep-15, Oct-15]",True,True,False,-89.0,-0.01,0.01,True
7,url,str,466285.0,466285.0,0.00,[https://www.lendingclub.com/browse/loanDetail...,str,421094,421094,0.00,[https://www.lendingclub.com/browse/loanDetail...,True,True,False,-45191.0,0.00,0.00,True
8,issue_d,str,91.0,91.0,0.00,"[Dec-11, Nov-11, Oct-11, Sep-11, Aug-11]",str,12,12,0.00,"[Dec-15, Nov-15, Oct-15, Sep-15, Aug-15]",True,True,False,-79.0,0.00,0.00,True
9,zip_code,str,888.0,888.0,0.00,"[860xx, 309xx, 606xx, 917xx, 972xx]",str,914,914,0.00,"[200xx, 462xx, 672xx, 460xx, 605xx]",True,True,False,26.0,0.00,0.00,True


In [18]:
reports["value_differences"]

,column_name,value,present_in
0,verification_status_joint,Not Verified,test_only
1,verification_status_joint,Source Verified,test_only
2,verification_status_joint,Verified,test_only
3,next_pymnt_d,Apr-08,train_only
4,next_pymnt_d,Apr-09,train_only
...,...,...,...
281,zip_code,507xx,test_only
282,zip_code,509xx,test_only
283,zip_code,520xx,test_only
284,zip_code,555xx,test_only


In [ ]:
# -----------------------------------------------------------------------------------
# Raw string alignment checkpoint (train vs test)
# Purpose: detect categorical drift, casing/whitespace issues, and dtype quirks
# before any transformations or column drops.
# -----------------------------------------------------------------------------------

export_reports = build_string_alignment_report(
	training_dataframe=df_raw_training_data,
	test_dataframe=df_raw_test_data,
	sample_size=5,
	top_k=30,
	drilldown_max_columns=10,
	drilldown_top_values_per_side=20,
	log_file=PROJECT_LOG_FILE,
	export_dir=audit_dir,          # e.g. Path("reports")
	export_base_name="string_alignment",
	export_tag="raw",              # or "clean", "feature_base"
	export_sample_values_as_json=True,
)

## Initial Data Inspection – Training Set (2007–2014)

The raw training dataset contains:

- Rows: **466,285**  
- Columns: **75**  
- Memory usage: **~389 MB**  
- Data types: **49 numeric**, **26 object**  
- Fully null columns: **17**  
- Constant columns: **2** (`policy_code`, `application_type`)  
- Columns with more than 50% null values: **4**   

### Structural Findings

The 17 fully null columns are removed in transformation. Columns without observations contain no recoverable signal and introduce unnecessary dimensionality.

Two columns (`policy_code`, `application_type`) exhibit zero variance and are likewise removed.

Three high-null credit timing variables  
(`mths_since_last_delinq`, `mths_since_last_major_derog`, `mths_since_last_record`) reflect structural absence rather than missing data. These are retained and restructured using:

- a binary indicator (event vs. no event)  
- a sentinel value (9999) to preserve ordinal meaning  

The remaining high-null column (`desc`) is an optional free-text field and is excluded to maintain a structured modeling scope.

No material mixed-type inconsistencies were detected. Object-encoded numerics and dates will be normalized during transformation.

Payment and recovery variables are present in the dataset but reflect post-origination information. These are explicitly excluded under the submission-time constraint.

---

Non-informative and ineligible variables are removed during transformation, after which the dataset is standardized and persisted in Parquet format.

## Feature Eligibility – Prediction at Application Submission

The prediction point is fixed at loan application submission.

This constraint governs feature eligibility. Only information observable at the moment the borrower submits the application may enter the training feature set.

Each variable is classified using the official LendingClub data dictionary and lifecycle timing. The objective is not to remove obvious leakage, but to align the feature space with the information set that would have existed in real time.

---

### Identifier Handling

A stable internal identifier (`row_id`) is created at the start of transformation. It exists solely to preserve row-level alignment across training, validation, and evaluation outputs.

Platform identifiers (`id`, `member_id`) and structural artifacts such as `Unnamed: 0` are removed. These fields carry no predictive signal and introduce unnecessary dimensionality.

---

### Eligible Variables

The retained feature space consists of two categories:

1. **Borrower-provided application data**  
   Income, employment characteristics, requested amount, purpose, home ownership, and related inputs declared at submission.

2. **Credit bureau snapshot variables**  
   FICO ranges, delinquency history, revolving utilization, open accounts, inquiry counts, and related credit metrics reflecting the borrower’s credit state at application time.

These variables collectively represent the borrower risk profile as observable at submission.

Variables reflecting later updates to credit state or platform processing steps are excluded. The model must not rely on information that becomes available only after internal review or loan issuance.

---

### Excluded Variables

Variables are excluded for one of four structural reasons:

1. **Underwriting and verification outcomes**  
   Fields such as `verification_status` reflect actions taken after submission. Including them would embed internal decision logic into the model.

2. **Platform pricing and internal risk assignments**  
   `int_rate`, `grade`, `sub_grade`, and `installment` are determined during underwriting and pricing. These variables represent LendingClub’s own risk assessment. Training on them would create circularity rather than independent inference.

3. **Workflow and servicing variables**  
   Listing status, payment history, recoveries, outstanding balances, and other servicing metrics are only known after origination. Their inclusion would constitute direct leakage.

4. **Structural and non-informative fields**  
   Identifiers, URLs, constant columns, and fully null columns are removed during ETL.

Each exclusion follows directly from the submission-time constraint rather than discretionary cleaning.

---

### Benchmark Variables

A limited subset of platform-assigned risk signals (`grade`, `sub_grade`, `int_rate`, `installment`) is retained in the cleaned dataset for benchmarking only.

These variables are never used for training. They serve as reference points during validation to compare the submission-time model against LendingClub’s origination-time assessment.

---

### Target Variable

`loan_status` defines the outcome and is excluded from the predictor space.

All ineligible variables are removed prior to EDA to ensure that subsequent analysis operates strictly within the defined information boundary.

## Feature Classification Overview – Application Submission Prediction Point

Feature eligibility follows three governing principles:

1. **Temporal availability** — The variable must be observable at the moment of application submission.
2. **Information independence** — The model must not embed LendingClub’s underwriting or pricing decisions.
3. **Structural viability** — The variable must contain usable information in the 2007–2014 training window.

Lifecycle timing and variable definitions are interpreted using the official LendingClub Data Dictionary that accompanies the dataset. Classification decisions therefore reflect documented field descriptions rather than inference from column names alone.

Each column is evaluated against these principles before inclusion or exclusion.

---

| Column | Category | Action | Rationale |
|--------|----------|--------|-----------|
| `annual_inc`, `dti` | Application input | Keep | Declared by borrower at submission |
| `loan_amnt`, `term`, `purpose` | Application input | Keep | Application-time inputs |
| `home_ownership`, `emp_length` | Application input | Keep | Observable at submission |
| `open_acc`, `total_acc` | Credit snapshot | Keep | Credit file state at application |
| `inq_last_6mths` | Credit snapshot | Keep | Recent inquiry activity |
| `delinq_2yrs` | Credit snapshot | Keep | Historical delinquency count |
| `pub_rec` | Credit snapshot | Keep | Public record count |
| `collections_12_mths_ex_med` | Credit snapshot | Keep | Recent collections history |
| `revol_bal`, `revol_util` | Credit snapshot | Keep | Revolving exposure and utilization |
| `acc_now_delinq` | Credit snapshot | Keep | Current delinquency indicator |
| `tot_cur_bal`, `tot_coll_amt` | Credit snapshot | Keep | Aggregate credit balances |
| `mths_since_last_delinq` | Credit timing | Keep (with transformation) | Null encodes structural absence; transformed with indicator + sentinel |
| `mths_since_last_major_derog` | Credit timing | Keep (with transformation) | Structural absence preserved via explicit encoding |
| `mths_since_last_record` | Credit timing | Keep (with transformation) | Structural absence preserved via explicit encoding |
| `loan_status` | Target | Keep (Target Only) | Defines outcome; excluded from predictors |
| `grade`, `sub_grade` | Platform risk signal | Retain (Benchmark Only) | Origination-time assessment; excluded from training to avoid circularity |
| `int_rate` | Pricing signal | Retain (Benchmark Only) | Underwriting output; retained only for benchmarking |
| `installment` | Pricing-derived | Retain (Benchmark Only) | Derived from pricing; excluded from training |
| `earliest_cr_line` | Credit history timestamp | Exclude (for now) | Absolute date requires application timestamp to derive credit age; proxying with `issue_d` would violate boundary |
| `verification_status`, `is_inc_v` | Underwriting outcome | Exclude | Determined during review process |
| `funded_amnt`, `funded_amnt_inv` | Funding decision | Exclude | Reflect platform allocation decision |
| `issue_d` | Lifecycle timestamp | Exclude | Occurs after submission |
| `initial_list_status` | Workflow variable | Exclude | Listing outcome, not borrower characteristic |
| `last_credit_pull_d` | Lifecycle update | Exclude | Updated after submission |
| `last_fico_range_high`, `last_fico_range_low` | Lifecycle update | Exclude | Subsequent bureau updates |
| `total_pymnt`, `total_pymnt_inv` | Servicing | Exclude | Cashflows realized after issuance |
| `total_rec_prncp`, `total_rec_int`, `total_rec_late_fee` | Servicing | Exclude | Post-origination cashflows |
| `recoveries`, `collection_recovery_fee` | Servicing | Exclude | Charge-off recovery information |
| `out_prncp`, `out_prncp_inv` | Servicing | Exclude | Outstanding balance after issuance |
| `last_pymnt_d`, `next_pymnt_d` | Servicing timeline | Exclude | Post-origination timing variables |
| `last_pymnt_amnt` | Servicing | Exclude | Payment received after issuance |
| `pymnt_plan` | Servicing decision | Exclude | Determined after loan issuance |
| `annual_inc_joint`, `dti_joint`, `verification_status_joint` | Structurally null (2007–2014) | Drop | No observations in training window |
| `open_acc_6m`, `open_il_6m`, `open_il_12m`, `open_il_24m` | Structurally null (2007–2014) | Drop | No observations in training window |
| `mths_since_rcnt_il`, `total_bal_il`, `il_util` | Structurally null (2007–2014) | Drop | No observations in training window |
| `open_rv_12m`, `open_rv_24m`, `all_util` | Structurally null (2007–2014) | Drop | No observations in training window |
| `inq_fi`, `total_cu_tl`, `inq_last_12m`, `max_bal_bc` | Structurally null (2007–2014) | Drop | No observations in training window |
| `id`, `member_id` | Identifier | Drop | Unique identifiers; no predictive value |
| `url` | Identifier | Drop | Platform listing reference |
| `Unnamed: 0` | Structural artifact | Drop | Exported index column |
| `policy_code`, `application_type` | Constant | Drop | Zero variance in training data |
| `desc`, `emp_title`, `title` | Free-text field | Drop | High cardinality; unstructured; outside structured modeling scope |
| `addr_state`, `zip_code` | Application input | Drop | Removed to reduce proxy discrimination risk and improve cross-jurisdiction portability |

---

**Future enhancement:**  
If a true application submission timestamp becomes available, `earliest_cr_line` can be converted into `credit_age_years` and reconsidered under the temporal availability rule.

## Initial Data Inspection – Test Set (2015)

The same structural inspection procedure is applied to the 2015 test dataset.

The objective is to verify that:

- Schema and column presence align with the training dataset  
- Datatypes follow the same assignments  
- Null structure does not introduce unseen patterns  
- No additional structural artifacts are present  

This step is not exploratory. It is a consistency check.

Any discrepancy between training and test must be identified before transformation rules are enforced. ETL logic is designed to operate deterministically across both partitions. If the raw structures diverge, the divergence must be resolved explicitly rather than absorbed silently during cleaning.

The test set is therefore inspected under identical criteria to ensure that downstream modeling operates on structurally aligned inputs.

## Test Set (2015) Inspection – Structural Consistency Check

The 2015 test dataset contains 74 columns versus 75 in the training dataset.  
The difference is the absence of `Unnamed: 0`, a CSV export artifact that is removed during transformation in any case. This discrepancy is operationally irrelevant.

A more substantive difference appears in a subset of credit bureau variables.  
In the 2007–2014 training window, 17 columns are entirely null, indicating that these features were not populated during that period. In the 2015 dataset, several of these same columns contain values, although with high missing rates.

Because the model is trained exclusively on 2007–2014 data, variables that contain no information in the training window cannot be learned. Incorporating them at prediction time would introduce features the model has never observed.

To preserve temporal consistency and prevent unseen-feature bias, these columns are removed from both training and test datasets.

Beyond these expected vintage-related differences in feature availability, the schema, datatypes, and overall structure of the 2015 dataset align with the training data. The same deterministic transformation rules are therefore applied to both partitions.

## Data Transformation and Cleaning

The training (2007–2014) and testing (2015) datasets are transformed under the feature eligibility and structural rules defined earlier.

Transformation is applied deterministically and identically across both partitions. The objective is schema identity prior to modeling.

The transformation phase performs the following operations:

- Removal of structural artifacts, constant columns, and all variables excluded under the submission-time constraint.  
- Explicit alignment of the test dataset to the training feature space. Variables that contain no information in the training window are removed from both partitions to prevent unseen-feature bias.  
- Creation of a stable internal `row_id` to preserve record-level traceability across modeling, validation, and evaluation outputs.  
- Structured encoding of credit timing variables (`mths_since_last_delinq`, `mths_since_last_major_derog`, `mths_since_last_record`). Because null represents structural absence rather than missing data, each variable is decomposed into:
  - a binary indicator (event vs. no prior event), and  
  - a sentinel value (9999) to preserve ordinal distance without collapsing absence into zero.  
- Normalization of remaining object-based fields after column pruning, including:
  - conversion of numeric values stored as strings into numeric types,  
  - conversion of date fields into `datetime` format,  
  - standardization of categorical labels to ensure consistent representation across partitions.  

No feature engineering beyond structural normalization is introduced at this stage.

---

### Output Datasets

Two dataset layers are materialized:

1. **`feature_base`** — submission-time eligible features plus the target variable. This dataset is used for EDA and model training.  
2. **`clean`** — structurally aligned dataset that retains benchmark variables (e.g., `grade`, `sub_grade`, `int_rate`, `installment`) for validation and comparative analysis.

The separation ensures that model training remains independent of platform-assigned risk signals while preserving those signals for controlled evaluation.

The resulting datasets are temporally aligned, structurally consistent, and reproducible.

In [16]:
# Create row identifier for training data
df_clean_training_data = create_row_identifier(
    df_raw_training_data, 
    id_column_name="row_id", 
    log_file=project_log_file)

NameError: name 'project_log_file' is not defined

In [ ]:
# Create row identifier for test data
df_clean_test_data = create_row_identifier(
    df_raw_test_data, 
    id_column_name="row_id", 
    log_file=project_log_file)

In [ ]:
# Parameters for sentinel encoding of credit history features.
credit_recency_columns = [
    "mths_since_last_delinq",
    "mths_since_last_record",
    "mths_since_last_major_derog"
]

SENTINEL_VALUE = 9999

# Columns to drop for clean_datasets
structurally_drop_columns = [
    "annual_inc_joint", "dti_joint", "verification_status_joint", "open_acc_6m", "open_il_6m", "open_il_12m", 
    "open_il_24m","mths_since_rcnt_il", "total_bal_il", "il_util","open_rv_12m", "open_rv_24m", "all_util",
    "inq_fi", "total_cu_tl", "inq_last_12m", "max_bal_bc","id", "member_id","url", "Unnamed: 0", "policy_code", 
    "application_type", "desc", "emp_title", "title", "addr_state", "zip_code"
]

#columns to drop for modeling datasets (after cleaning)
excluded_columns = [
    "verification_status", "funded_amnt", "funded_amnt_inv", "issue_d", "initial_list_status",
    "last_credit_pull_d", "total_pymnt", "total_pymnt_inv", "total_rec_prncp", "total_rec_int", 
    "total_rec_late_fee", "recoveries", "collection_recovery_fee", "earliest_cr_line", "out_prncp", 
    "out_prncp_inv", "last_pymnt_d", "next_pymnt_d", "last_pymnt_amnt", "pymnt_plan"
]

benchmark_columns = ["grade", "sub_grade", "int_rate", "installment"]



In [ ]:
# Apply sentinel encoding to the credit recency columns in the training data
df_clean_training_data = apply_credit_sentinel_encoding(
    df_clean_training_data,
    credit_recency_columns,
    sentinel_value=SENTINEL_VALUE,
    log_file=project_log_file)

In [ ]:
# Apply sentinel encoding to the credit recency columns in the test data
df_clean_test_data = apply_credit_sentinel_encoding(
    df_clean_test_data,
    credit_recency_columns,
    sentinel_value=SENTINEL_VALUE,
    log_file=project_log_file)

In [ ]:
# Drop columns for cleaning training dataset
df_clean_training_data = drop_columns_with_logging(
    dataframe=df_clean_training_data,
    columns_to_drop=structurally_drop_columns,
    dataset_name="train_clean",
    log_file=project_log_file,
    errors='ignore'
    )

df_clean_training_data.head(5)

,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,...,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog
0,1,5000,5000,4975.0,36 months,10.65,162.87,B,B2,10+ years,...,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
1,2,2500,2500,2500.0,60 months,15.27,59.83,C,C4,< 1 year,...,Sep-13,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
2,3,2400,2400,2400.0,36 months,15.96,84.33,C,C5,10+ years,...,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
3,4,10000,10000,10000.0,36 months,13.49,339.31,C,C1,10+ years,...,Jan-15,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0
4,5,3000,3000,3000.0,60 months,12.69,67.79,B,B5,1 year,...,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0


In [ ]:
# Drop columns for cleaning test dataset
df_clean_test_data = drop_columns_with_logging(
    dataframe=df_clean_test_data,
    columns_to_drop=structurally_drop_columns,
    dataset_name="test_clean",
    log_file=project_log_file,
    errors='ignore'
    )

df_clean_test_data.head(5)

,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,...,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog
0,1,35000,35000,35000.0,60 months,11.99,778.38,C,C1,10+ years,...,Jan-16,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1
1,2,8650,8650,8650.0,36 months,5.32,260.50,A,A1,< 1 year,...,Jan-16,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0
2,3,4225,4225,4225.0,36 months,14.85,146.16,C,C5,5 years,...,Jan-16,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0
3,4,10000,10000,10000.0,60 months,11.99,222.40,C,C1,10+ years,...,Jan-16,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0
4,5,20000,20000,20000.0,60 months,10.78,432.66,B,B4,10+ years,...,Dec-15,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0


In [ ]:
# Inspecting string columns in the clean datasets
training_string_audit = audit_string_columns(
    df_clean_training_data,
    sample_size=15,
    log_file=project_log_file
)

test_string_audit = audit_string_columns(
    df_clean_test_data,
    sample_size=15,
    log_file=project_log_file
)

# Display training string audit
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    
    display(
        training_string_audit.sort_values("unique_count_non_null", ascending=False)
    )

    
# Display test string audit
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    
    display(
        test_string_audit.sort_values("unique_count_non_null", ascending=False)
    )


,column_name,dtype,unique_count_including_null,unique_count_non_null,null_percent,sample_values
0,earliest_cr_line,str,665,664,0.01,"[Jan-85, Apr-99, Nov-01, Feb-96, Jan-96, Nov-0..."
1,last_credit_pull_d,str,104,103,0.01,"[Jan-16, Sep-13, Jan-15, Sep-15, Dec-14, Aug-1..."
2,next_pymnt_d,str,101,100,48.73,"[Feb-16, Jan-16, Sep-13, Feb-14, May-14, Jun-1..."
3,last_pymnt_d,str,99,98,0.08,"[Jan-15, Apr-13, Jun-14, Jan-16, Apr-12, Nov-1..."
4,issue_d,str,91,91,0.00,"[Dec-11, Nov-11, Oct-11, Sep-11, Aug-11, Jul-1..."
5,sub_grade,str,35,35,0.00,"[B2, C4, C5, C1, B5, A4, E1, F2, C3, B1, D1, A..."
6,purpose,str,14,14,0.00,"[credit_card, car, small_business, other, wedd..."
7,emp_length,str,12,11,4.51,"[10+ years, < 1 year, 1 year, 3 years, 8 years..."
8,loan_status,str,9,9,0.00,"[Fully Paid, Charged Off, Current, Default, La..."
9,grade,str,7,7,0.00,"[B, C, A, E, F, D, G]"


,column_name,dtype,unique_count_including_null,unique_count_non_null,null_percent,sample_values
0,earliest_cr_line,str,668,668,0.00,"[Feb-90, Jul-01, Jul-11, Dec-98, Aug-00, Nov-0..."
1,sub_grade,str,35,35,0.00,"[C1, A1, C5, B4, B3, A4, C4, C3, F1, D3, B1, E..."
2,purpose,str,14,14,0.00,"[home_improvement, credit_card, debt_consolida..."
3,last_credit_pull_d,str,15,14,0.00,"[Jan-16, Dec-15, Nov-15, Sep-15, Oct-15, Aug-1..."
4,last_pymnt_d,str,14,13,4.10,"[Jan-16, Dec-15, Nov-15, Oct-15, Sep-15, Aug-1..."
5,issue_d,str,12,12,0.00,"[Dec-15, Nov-15, Oct-15, Sep-15, Aug-15, Jul-1..."
6,emp_length,str,12,11,5.66,"[10+ years, < 1 year, 5 years, 3 years, 4 year..."
7,loan_status,str,8,8,0.00,"[Issued, Current, Fully Paid, In Grace Period,..."
8,grade,str,7,7,0.00,"[C, A, B, F, D, E, G]"
9,home_ownership,str,4,4,0.00,"[MORTGAGE, RENT, OWN, ANY]"


In [ ]:
# Compare the string columns between training and test datasets
audit_columns_to_compare = [
    "column_name",
    "dtype",
    "unique_count_non_null",
    "unique_count_including_null",
    "null_percent",
]

training_audit_subset = training_string_audit[audit_columns_to_compare].copy()
test_audit_subset = test_string_audit[audit_columns_to_compare].copy()

training_audit_subset = training_audit_subset.rename(columns={
    "dtype": "dtype_train",
    "unique_count_non_null": "unique_non_null_train",
    "unique_count_including_null": "unique_including_null_train",
    "null_percent": "null_percent_train",
})

test_audit_subset = test_audit_subset.rename(columns={
    "dtype": "dtype_test",
    "unique_count_non_null": "unique_non_null_test",
    "unique_count_including_null": "unique_including_null_test",
    "null_percent": "null_percent_test",
})

string_audit_comparison = (
    training_audit_subset
    .merge(test_audit_subset, on="column_name", how="outer")
    .sort_values(by="column_name")
    .reset_index(drop=True)
)

string_audit_comparison["unique_non_null_diff"] = (
    string_audit_comparison["unique_non_null_train"].fillna(0).astype(int)
    - string_audit_comparison["unique_non_null_test"].fillna(0).astype(int)
)

string_audit_comparison


,column_name,dtype_train,unique_non_null_train,unique_including_null_train,null_percent_train,dtype_test,unique_non_null_test,unique_including_null_test,null_percent_test,unique_non_null_diff
0,earliest_cr_line,str,664,665,0.01,str,668,668,0.00,-4
1,emp_length,str,11,12,4.51,str,11,12,5.66,0
2,grade,str,7,7,0.00,str,7,7,0.00,0
3,home_ownership,str,6,6,0.00,str,4,4,0.00,2
4,initial_list_status,str,2,2,0.00,str,2,2,0.00,0
5,issue_d,str,91,91,0.00,str,12,12,0.00,79
6,last_credit_pull_d,str,103,104,0.01,str,14,15,0.00,89
7,last_pymnt_d,str,98,99,0.08,str,13,14,4.10,85
8,loan_status,str,9,9,0.00,str,8,8,0.00,1
9,next_pymnt_d,str,100,101,48.73,str,3,4,6.12,97


In [ ]:
string_columns_with_differences = string_audit_comparison[
    (string_audit_comparison["dtype_train"] != string_audit_comparison["dtype_test"])
    | (string_audit_comparison["unique_non_null_train"] != string_audit_comparison["unique_non_null_test"])
    | (string_audit_comparison["null_percent_train"].round(2) != string_audit_comparison["null_percent_test"].round(2))
].copy()

string_columns_with_differences


,column_name,dtype_train,unique_non_null_train,unique_including_null_train,null_percent_train,dtype_test,unique_non_null_test,unique_including_null_test,null_percent_test,unique_non_null_diff
0,earliest_cr_line,str,664,665,0.01,str,668,668,0.00,-4
1,emp_length,str,11,12,4.51,str,11,12,5.66,0
3,home_ownership,str,6,6,0.00,str,4,4,0.00,2
5,issue_d,str,91,91,0.00,str,12,12,0.00,79
6,last_credit_pull_d,str,103,104,0.01,str,14,15,0.00,89
7,last_pymnt_d,str,98,99,0.08,str,13,14,4.10,85
8,loan_status,str,9,9,0.00,str,8,8,0.00,1
9,next_pymnt_d,str,100,101,48.73,str,3,4,6.12,97


In [ ]:
columns_to_drill_down = (
    string_columns_with_differences["column_name"]
    .dropna()
    .astype(str)
    .tolist()
)

categorical_value_differences = {}

for column_name in columns_to_drill_down:
    categorical_value_differences[column_name] = compare_categorical_column_values(
        training_dataframe=df_clean_training_data,
        test_dataframe=df_clean_test_data,
        column_name=column_name,
        log_file=project_log_file,
    )


In [ ]:
summary_records = []

for column_name in columns_to_drill_down:
    comparison_dataframe = compare_categorical_column_values(
        training_dataframe=df_clean_training_data,
        test_dataframe=df_clean_test_data,
        column_name=column_name,
        log_file=project_log_file,
    )

    categorical_value_differences[column_name] = comparison_dataframe

    missing_in_training = (~comparison_dataframe["present_in_training"]).sum()
    missing_in_test = (~comparison_dataframe["present_in_test"]).sum()

    summary_records.append({
        "column_name": column_name,
        "categories_total": len(comparison_dataframe),
        "only_in_test": int(missing_in_training),
        "only_in_training": int(missing_in_test),
    })

df_category_diff_summary = (
    pd.DataFrame(summary_records)
    .sort_values(["only_in_test", "only_in_training"], ascending=False)
    .reset_index(drop=True)
)

df_category_diff_summary


,column_name,categories_total,only_in_test,only_in_training
0,earliest_cr_line,697,33,29
1,issue_d,103,12,91
2,loan_status,10,1,2
3,next_pymnt_d,100,0,97
4,last_credit_pull_d,103,0,89
5,last_pymnt_d,98,0,85
6,home_ownership,6,0,2
7,emp_length,11,0,0


In [ ]:
# Log categorical differences across training and test datasets
categorical_value_differences = {}

for column_name in columns_to_drill_down:
    categorical_value_differences[column_name] = compare_categorical_column_values(
        training_dataframe=df_clean_training_data,
        test_dataframe=df_clean_test_data,
        column_name=column_name,
        log_file=PROJECT_LOG_FILE,
    )

log_category_differences(
    categorical_value_differences=categorical_value_differences,
    log_file=PROJECT_LOG_FILE,
)


KeyError: 'category'

## String Column Treatment Overview

After structural column removal, the remaining object-based fields are reviewed individually.

The objective is not cosmetic cleaning, but datatype normalization and boundary enforcement.  
Each string column is evaluated along two dimensions:

- **Transformation action** — how the raw representation is converted into a stable analytical form.  
- **Training eligibility** — whether the variable is part of the submission-time feature space, retained only for benchmarking, or excluded.

Observed category differences between the 2007–2014 training window and the 2015 test window reflect temporal coverage rather than schema drift. No additional harmonization beyond normalization is required.

---

| Column | Category | Transformation Action | Training Eligibility | Rationale |
|---------|------------|------------------------|----------------------|------------|
| `term` | Structured categorical | Strip whitespace → extract numeric term (36 / 60) → rename to `term_months` → convert to int | Keep | Contractual term declared at submission |
| `emp_length` | Ordinal categorical | Map to integer scale 0–10 (`<1` → 0, `10+` → 10); retain NaN | Keep | Missing indicates unknown tenure, not structural absence |
| `home_ownership` | Categorical | Standardize casing; map `NONE` → `OTHER`; cast to category | Keep | Borrower-declared attribute |
| `purpose` | Categorical | Standardize casing; cast to category | Keep | Borrower-declared loan purpose |
| `grade` | Platform risk signal | Standardize casing; cast to category | Retain (Benchmark Only) | Assigned during underwriting |
| `sub_grade` | Platform risk signal | Standardize casing; cast to category | Retain (Benchmark Only) | Granular underwriting signal |
| `verification_status` | Underwriting outcome | Standardize casing; cast to category | Exclude | Determined during review process |
| `initial_list_status` | Workflow variable | Standardize casing; cast to category | Exclude | Platform listing outcome |
| `pymnt_plan` | Servicing indicator | Map `n` → 0, `y` → 1 | Exclude | Post-origination servicing flag |
| `issue_d` | Lifecycle timestamp | Convert month-year string to datetime | Exclude | Occurs after submission |
| `last_credit_pull_d` | Lifecycle update | Convert month-year string to datetime | Exclude | Subsequent bureau update |
| `last_pymnt_d` | Servicing timeline | Convert month-year string to datetime | Exclude | Post-origination variable |
| `next_pymnt_d` | Servicing timeline | Convert month-year string to datetime | Exclude | Post-origination variable |
| `loan_status` | Target | No transformation at this stage | Keep (Target Only) | Outcome definition; cohort defined later |

---

### Structural Notes

All categorical normalization is applied identically across training and test partitions. No category collapsing or frequency-based pruning is introduced at this stage.

Datetime conversions are performed for structural consistency only; lifecycle variables remain excluded under the submission-time constraint.

The resulting object-based feature space is normalized, typed, and aligned across partitions.

In [ ]:
# -----------------------------------
# Columns grouped by transformation type
# -----------------------------------

# 1. Text normalization (strip + lowercase + cast to category)
# Used to standardize categorical values and ensure train/test alignment.
categorical_normalization_columns = [
    "home_ownership",
    "purpose",
    "grade",
    "sub_grade",
    "verification_status",
    "initial_list_status",
    "loan_status",
]

# 2. Deterministic category remapping
# Used to consolidate semantically equivalent labels.
special_categorical_mappings = {
    "home_ownership": {
        "none": "other",
        "any": "other",
    }
}

# 3. Ordinal encoding
# Used for ordered categories with intrinsic rank.
ordinal_encoding_columns = {
    "emp_length": {
        "< 1 year": 0,
        "1 year": 1,
        "2 years": 2,
        "3 years": 3,
        "4 years": 4,
        "5 years": 5,
        "6 years": 6,
        "7 years": 7,
        "8 years": 8,
        "9 years": 9,
        "10+ years": 10
    }
}

# 4. Structured extraction
# Convert contract term to numeric month representation.
term_column = "term"
term_new_name = "term_months"

# 5. Binary encoding
# Convert binary categorical indicators to numeric form.
binary_encoding_columns = {
    "pymnt_plan": {
        "n": 0,
        "y": 1
    }
}

# 6. Month-Year datetime conversion
# Convert month-year strings to datetime format.
month_year_datetime_columns = [
    "issue_d",
    "last_credit_pull_d",
    "last_pymnt_d",
    "next_pymnt_d",
    "earliest_cr_line",
]

# 7. Explicit mapping for listing type
# Replace abbreviated codes with semantic labels.
initial_list_status_mapping = {
    "w": "whole",
    "f": "fractional"
}


In [ ]:
# 1. Normalize text categories (casing/whitespace)
df_clean_training_data = normalize_text_columns(
    dataframe=df_clean_training_data,
    columns_to_normalize=categorical_normalization_columns,
    log_file=PROJECT_LOG_FILE,
)

df_clean_test_data = normalize_text_columns(
    dataframe=df_clean_test_data,
    columns_to_normalize=categorical_normalization_columns,
    log_file=PROJECT_LOG_FILE,
)

print(
    f"Train shape: {df_clean_training_data.shape} | "
    f"Test shape: {df_clean_test_data.shape}"
)

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_clean_training_data.head(5))
    display(df_clean_test_data.head(5))

display(df_clean_training_data.dtypes)
display(df_clean_test_data.dtypes)

Train shape: (466285, 51) | Test shape: (421094, 51)


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog
0,1,5000,5000,4975.0,36 months,10.65,162.87,b,b2,10+ years,rent,24000.0,verified,Dec-11,fully_paid,n,credit_card,27.65,0.0,Jan-85,1.0,9999.0,9999.0,3.0,0.0,13648,83.7,9.0,f,0.0,0.0,5861.071414,5831.78,5000.00,861.07,0.00,0.00,0.00,Jan-15,171.62,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
1,2,2500,2500,2500.0,60 months,15.27,59.83,c,c4,< 1 year,rent,30000.0,source_verified,Dec-11,charged_off,n,car,1.00,0.0,Apr-99,5.0,9999.0,9999.0,3.0,0.0,1687,9.4,4.0,f,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,Apr-13,119.66,NaN,Sep-13,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
2,3,2400,2400,2400.0,36 months,15.96,84.33,c,c5,10+ years,rent,12252.0,not_verified,Dec-11,fully_paid,n,small_business,8.72,0.0,Nov-01,2.0,9999.0,9999.0,2.0,0.0,2956,98.5,10.0,f,0.0,0.0,3003.653644,3003.65,2400.00,603.65,0.00,0.00,0.00,Jun-14,649.91,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
3,4,10000,10000,10000.0,36 months,13.49,339.31,c,c1,10+ years,rent,49200.0,source_verified,Dec-11,fully_paid,n,other,20.00,0.0,Feb-96,1.0,35.0,9999.0,10.0,0.0,5598,21.0,37.0,f,0.0,0.0,12226.302210,12226.30,10000.00,2209.33,16.97,0.00,0.00,Jan-15,357.48,NaN,Jan-15,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0
4,5,3000,3000,3000.0,60 months,12.69,67.79,b,b5,1 year,rent,80000.0,source_verified,Dec-11,current,n,other,17.94,0.0,Jan-96,0.0,38.0,9999.0,15.0,0.0,27783,53.9,38.0,f,766.9,766.9,3242.170000,3242.17,2233.10,1009.07,0.00,0.00,0.00,Jan-16,67.79,Feb-16,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog
0,1,35000,35000,35000.0,60 months,11.99,778.38,c,c1,10+ years,mortgage,128000.0,source_verified,Dec-15,issued,n,home_improvement,6.46,0.0,Feb-90,0.0,46.0,9999.0,17.0,0.0,14277,27.4,46.0,w,35000.0,35000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1
1,2,8650,8650,8650.0,36 months,5.32,260.50,a,a1,< 1 year,mortgage,100000.0,not_verified,Dec-15,issued,n,credit_card,7.28,0.0,Jul-01,0.0,9999.0,9999.0,15.0,0.0,7158,26.7,24.0,w,8650.0,8650.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0
2,3,4225,4225,4225.0,36 months,14.85,146.16,c,c5,5 years,rent,35000.0,source_verified,Dec-15,issued,n,debt_consolidation,15.22,2.0,Jul-11,0.0,18.0,9999.0,6.0,0.0,1058,24.6,6.0,w,4225.0,4225.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0
3,4,10000,10000,10000.0,60 months,11.99,222.40,c,c1,10+ years,rent,42500.0,not_verified,Dec-15,issued,n,debt_consolidation,31.04,0.0,Dec-98,1.0,9999.0,9999.0,10.0,0.0,5812,40.9,23.0,w,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0
4,5,20000,20000,20000.0,60 months,10.78,432.66,b,b4,10+ years,mortgage,63000.0,not_verified,Dec-15,issued,n,home_improvement,10.78,0.0,Aug-00,0.0,9999.0,9999.0,6.0,0.0,7869,56.2,18.0,w,20000.0,20000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Dec-15,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0


row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                         str
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                             str
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                         str
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                             str
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

In [ ]:
# 2. Home ownership alignment
df_clean_training_data = normalize_home_ownership(df_clean_training_data, log_file=PROJECT_LOG_FILE)
df_clean_test_data = normalize_home_ownership(df_clean_test_data, log_file=PROJECT_LOG_FILE)

print(
    f"Train shape: {df_clean_training_data.shape} | "
    f"Test shape: {df_clean_test_data.shape}"
)

print(
    f"Train shape: {df_clean_training_data.shape} | "
    f"Test shape: {df_clean_test_data.shape}"
)

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_clean_training_data.head(5))
    display(df_clean_test_data.head(5))

display(df_clean_training_data.dtypes)
display(df_clean_test_data.dtypes)

Train shape: (466285, 51) | Test shape: (421094, 51)
Train shape: (466285, 51) | Test shape: (421094, 51)


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog
0,1,5000,5000,4975.0,36 months,10.65,162.87,b,b2,10+ years,rent,24000.0,verified,Dec-11,fully_paid,n,credit_card,27.65,0.0,Jan-85,1.0,9999.0,9999.0,3.0,0.0,13648,83.7,9.0,f,0.0,0.0,5861.071414,5831.78,5000.00,861.07,0.00,0.00,0.00,Jan-15,171.62,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
1,2,2500,2500,2500.0,60 months,15.27,59.83,c,c4,< 1 year,rent,30000.0,source_verified,Dec-11,charged_off,n,car,1.00,0.0,Apr-99,5.0,9999.0,9999.0,3.0,0.0,1687,9.4,4.0,f,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,Apr-13,119.66,NaN,Sep-13,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
2,3,2400,2400,2400.0,36 months,15.96,84.33,c,c5,10+ years,rent,12252.0,not_verified,Dec-11,fully_paid,n,small_business,8.72,0.0,Nov-01,2.0,9999.0,9999.0,2.0,0.0,2956,98.5,10.0,f,0.0,0.0,3003.653644,3003.65,2400.00,603.65,0.00,0.00,0.00,Jun-14,649.91,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
3,4,10000,10000,10000.0,36 months,13.49,339.31,c,c1,10+ years,rent,49200.0,source_verified,Dec-11,fully_paid,n,other,20.00,0.0,Feb-96,1.0,35.0,9999.0,10.0,0.0,5598,21.0,37.0,f,0.0,0.0,12226.302210,12226.30,10000.00,2209.33,16.97,0.00,0.00,Jan-15,357.48,NaN,Jan-15,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0
4,5,3000,3000,3000.0,60 months,12.69,67.79,b,b5,1 year,rent,80000.0,source_verified,Dec-11,current,n,other,17.94,0.0,Jan-96,0.0,38.0,9999.0,15.0,0.0,27783,53.9,38.0,f,766.9,766.9,3242.170000,3242.17,2233.10,1009.07,0.00,0.00,0.00,Jan-16,67.79,Feb-16,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog
0,1,35000,35000,35000.0,60 months,11.99,778.38,c,c1,10+ years,mortgage,128000.0,source_verified,Dec-15,issued,n,home_improvement,6.46,0.0,Feb-90,0.0,46.0,9999.0,17.0,0.0,14277,27.4,46.0,w,35000.0,35000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1
1,2,8650,8650,8650.0,36 months,5.32,260.50,a,a1,< 1 year,mortgage,100000.0,not_verified,Dec-15,issued,n,credit_card,7.28,0.0,Jul-01,0.0,9999.0,9999.0,15.0,0.0,7158,26.7,24.0,w,8650.0,8650.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0
2,3,4225,4225,4225.0,36 months,14.85,146.16,c,c5,5 years,rent,35000.0,source_verified,Dec-15,issued,n,debt_consolidation,15.22,2.0,Jul-11,0.0,18.0,9999.0,6.0,0.0,1058,24.6,6.0,w,4225.0,4225.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0
3,4,10000,10000,10000.0,60 months,11.99,222.40,c,c1,10+ years,rent,42500.0,not_verified,Dec-15,issued,n,debt_consolidation,31.04,0.0,Dec-98,1.0,9999.0,9999.0,10.0,0.0,5812,40.9,23.0,w,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0
4,5,20000,20000,20000.0,60 months,10.78,432.66,b,b4,10+ years,mortgage,63000.0,not_verified,Dec-15,issued,n,home_improvement,10.78,0.0,Aug-00,0.0,9999.0,9999.0,6.0,0.0,7869,56.2,18.0,w,20000.0,20000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Dec-15,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0


row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                      string
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                             str
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                      string
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                             str
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

In [ ]:
# 3. initial_list_status mapping (w/f -> words)
df_clean_training_data = apply_categorical_mapping(
    dataframe=df_clean_training_data,
    column_name="initial_list_status",
    mapping=initial_list_status_mapping,
    log_file=project_log_file,
    allow_unmapped_values=False,
)

df_clean_test_data = apply_categorical_mapping(
    dataframe=df_clean_test_data,
    column_name="initial_list_status",
    mapping=initial_list_status_mapping,
    log_file=project_log_file,
    allow_unmapped_values=False,
)

print(
    f"Train shape: {df_clean_training_data.shape} | "
    f"Test shape: {df_clean_test_data.shape}"
)

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_clean_training_data.head(5))
    display(df_clean_test_data.head(5))

display(df_clean_training_data.dtypes)
display(df_clean_test_data.dtypes)

Train shape: (466285, 51) | Test shape: (421094, 51)


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog
0,1,5000,5000,4975.0,36 months,10.65,162.87,b,b2,10+ years,rent,24000.0,verified,Dec-11,fully_paid,n,credit_card,27.65,0.0,Jan-85,1.0,9999.0,9999.0,3.0,0.0,13648,83.7,9.0,fractional,0.0,0.0,5861.071414,5831.78,5000.00,861.07,0.00,0.00,0.00,Jan-15,171.62,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
1,2,2500,2500,2500.0,60 months,15.27,59.83,c,c4,< 1 year,rent,30000.0,source_verified,Dec-11,charged_off,n,car,1.00,0.0,Apr-99,5.0,9999.0,9999.0,3.0,0.0,1687,9.4,4.0,fractional,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,Apr-13,119.66,NaN,Sep-13,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
2,3,2400,2400,2400.0,36 months,15.96,84.33,c,c5,10+ years,rent,12252.0,not_verified,Dec-11,fully_paid,n,small_business,8.72,0.0,Nov-01,2.0,9999.0,9999.0,2.0,0.0,2956,98.5,10.0,fractional,0.0,0.0,3003.653644,3003.65,2400.00,603.65,0.00,0.00,0.00,Jun-14,649.91,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
3,4,10000,10000,10000.0,36 months,13.49,339.31,c,c1,10+ years,rent,49200.0,source_verified,Dec-11,fully_paid,n,other,20.00,0.0,Feb-96,1.0,35.0,9999.0,10.0,0.0,5598,21.0,37.0,fractional,0.0,0.0,12226.302210,12226.30,10000.00,2209.33,16.97,0.00,0.00,Jan-15,357.48,NaN,Jan-15,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0
4,5,3000,3000,3000.0,60 months,12.69,67.79,b,b5,1 year,rent,80000.0,source_verified,Dec-11,current,n,other,17.94,0.0,Jan-96,0.0,38.0,9999.0,15.0,0.0,27783,53.9,38.0,fractional,766.9,766.9,3242.170000,3242.17,2233.10,1009.07,0.00,0.00,0.00,Jan-16,67.79,Feb-16,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog
0,1,35000,35000,35000.0,60 months,11.99,778.38,c,c1,10+ years,mortgage,128000.0,source_verified,Dec-15,issued,n,home_improvement,6.46,0.0,Feb-90,0.0,46.0,9999.0,17.0,0.0,14277,27.4,46.0,whole,35000.0,35000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1
1,2,8650,8650,8650.0,36 months,5.32,260.50,a,a1,< 1 year,mortgage,100000.0,not_verified,Dec-15,issued,n,credit_card,7.28,0.0,Jul-01,0.0,9999.0,9999.0,15.0,0.0,7158,26.7,24.0,whole,8650.0,8650.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0
2,3,4225,4225,4225.0,36 months,14.85,146.16,c,c5,5 years,rent,35000.0,source_verified,Dec-15,issued,n,debt_consolidation,15.22,2.0,Jul-11,0.0,18.0,9999.0,6.0,0.0,1058,24.6,6.0,whole,4225.0,4225.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0
3,4,10000,10000,10000.0,60 months,11.99,222.40,c,c1,10+ years,rent,42500.0,not_verified,Dec-15,issued,n,debt_consolidation,31.04,0.0,Dec-98,1.0,9999.0,9999.0,10.0,0.0,5812,40.9,23.0,whole,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0
4,5,20000,20000,20000.0,60 months,10.78,432.66,b,b4,10+ years,mortgage,63000.0,not_verified,Dec-15,issued,n,home_improvement,10.78,0.0,Aug-00,0.0,9999.0,9999.0,6.0,0.0,7869,56.2,18.0,whole,20000.0,20000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Dec-15,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0


row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                      string
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                             str
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                      string
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                             str
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

In [ ]:
# 4. pymnt_plan binary
df_clean_training_data = apply_categorical_mapping(
    dataframe=df_clean_training_data,
    column_name="pymnt_plan",
    mapping=binary_encoding_columns["pymnt_plan"],
    log_file=project_log_file,
    allow_unmapped_values=False,
)

df_clean_test_data = apply_categorical_mapping(
    dataframe=df_clean_test_data,
    column_name="pymnt_plan",
    mapping=binary_encoding_columns["pymnt_plan"],
    log_file=project_log_file,
    allow_unmapped_values=False,
)

print(
    f"Train shape: {df_clean_training_data.shape} | "
    f"Test shape: {df_clean_test_data.shape}"
)

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_clean_training_data.head(5))
    display(df_clean_test_data.head(5))

display(df_clean_training_data.dtypes)
display(df_clean_test_data.dtypes)

Train shape: (466285, 51) | Test shape: (421094, 51)


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog
0,1,5000,5000,4975.0,36 months,10.65,162.87,b,b2,10+ years,rent,24000.0,verified,Dec-11,fully_paid,0,credit_card,27.65,0.0,Jan-85,1.0,9999.0,9999.0,3.0,0.0,13648,83.7,9.0,fractional,0.0,0.0,5861.071414,5831.78,5000.00,861.07,0.00,0.00,0.00,Jan-15,171.62,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
1,2,2500,2500,2500.0,60 months,15.27,59.83,c,c4,< 1 year,rent,30000.0,source_verified,Dec-11,charged_off,0,car,1.00,0.0,Apr-99,5.0,9999.0,9999.0,3.0,0.0,1687,9.4,4.0,fractional,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,Apr-13,119.66,NaN,Sep-13,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
2,3,2400,2400,2400.0,36 months,15.96,84.33,c,c5,10+ years,rent,12252.0,not_verified,Dec-11,fully_paid,0,small_business,8.72,0.0,Nov-01,2.0,9999.0,9999.0,2.0,0.0,2956,98.5,10.0,fractional,0.0,0.0,3003.653644,3003.65,2400.00,603.65,0.00,0.00,0.00,Jun-14,649.91,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
3,4,10000,10000,10000.0,36 months,13.49,339.31,c,c1,10+ years,rent,49200.0,source_verified,Dec-11,fully_paid,0,other,20.00,0.0,Feb-96,1.0,35.0,9999.0,10.0,0.0,5598,21.0,37.0,fractional,0.0,0.0,12226.302210,12226.30,10000.00,2209.33,16.97,0.00,0.00,Jan-15,357.48,NaN,Jan-15,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0
4,5,3000,3000,3000.0,60 months,12.69,67.79,b,b5,1 year,rent,80000.0,source_verified,Dec-11,current,0,other,17.94,0.0,Jan-96,0.0,38.0,9999.0,15.0,0.0,27783,53.9,38.0,fractional,766.9,766.9,3242.170000,3242.17,2233.10,1009.07,0.00,0.00,0.00,Jan-16,67.79,Feb-16,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog
0,1,35000,35000,35000.0,60 months,11.99,778.38,c,c1,10+ years,mortgage,128000.0,source_verified,Dec-15,issued,0,home_improvement,6.46,0.0,Feb-90,0.0,46.0,9999.0,17.0,0.0,14277,27.4,46.0,whole,35000.0,35000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1
1,2,8650,8650,8650.0,36 months,5.32,260.50,a,a1,< 1 year,mortgage,100000.0,not_verified,Dec-15,issued,0,credit_card,7.28,0.0,Jul-01,0.0,9999.0,9999.0,15.0,0.0,7158,26.7,24.0,whole,8650.0,8650.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0
2,3,4225,4225,4225.0,36 months,14.85,146.16,c,c5,5 years,rent,35000.0,source_verified,Dec-15,issued,0,debt_consolidation,15.22,2.0,Jul-11,0.0,18.0,9999.0,6.0,0.0,1058,24.6,6.0,whole,4225.0,4225.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0
3,4,10000,10000,10000.0,60 months,11.99,222.40,c,c1,10+ years,rent,42500.0,not_verified,Dec-15,issued,0,debt_consolidation,31.04,0.0,Dec-98,1.0,9999.0,9999.0,10.0,0.0,5812,40.9,23.0,whole,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0
4,5,20000,20000,20000.0,60 months,10.78,432.66,b,b4,10+ years,mortgage,63000.0,not_verified,Dec-15,issued,0,home_improvement,10.78,0.0,Aug-00,0.0,9999.0,9999.0,6.0,0.0,7869,56.2,18.0,whole,20000.0,20000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Dec-15,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0


row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                      string
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                           int64
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                      string
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                           int64
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

In [ ]:
# 5. Term parsing + rename
# (Implement as a dedicated function if you want logging; otherwise do it inline)
df_clean_training_data[term_new_name] = (
    df_clean_training_data[term_column].astype("string").str.strip().str.extract(r"(\d+)")[0].astype("Int16")
)
df_clean_test_data[term_new_name] = (
    df_clean_test_data[term_column].astype("string").str.strip().str.extract(r"(\d+)")[0].astype("Int16")
)

print(
    f"Train shape: {df_clean_training_data.shape} | "
    f"Test shape: {df_clean_test_data.shape}"
)

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_clean_training_data.head(5))
    display(df_clean_test_data.head(5))

display(df_clean_training_data.dtypes)
display(df_clean_test_data.dtypes)

Train shape: (466285, 52) | Test shape: (421094, 52)


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months
0,1,5000,5000,4975.0,36 months,10.65,162.87,b,b2,10+ years,rent,24000.0,verified,Dec-11,fully_paid,0,credit_card,27.65,0.0,Jan-85,1.0,9999.0,9999.0,3.0,0.0,13648,83.7,9.0,fractional,0.0,0.0,5861.071414,5831.78,5000.00,861.07,0.00,0.00,0.00,Jan-15,171.62,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36
1,2,2500,2500,2500.0,60 months,15.27,59.83,c,c4,< 1 year,rent,30000.0,source_verified,Dec-11,charged_off,0,car,1.00,0.0,Apr-99,5.0,9999.0,9999.0,3.0,0.0,1687,9.4,4.0,fractional,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,Apr-13,119.66,NaN,Sep-13,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,60
2,3,2400,2400,2400.0,36 months,15.96,84.33,c,c5,10+ years,rent,12252.0,not_verified,Dec-11,fully_paid,0,small_business,8.72,0.0,Nov-01,2.0,9999.0,9999.0,2.0,0.0,2956,98.5,10.0,fractional,0.0,0.0,3003.653644,3003.65,2400.00,603.65,0.00,0.00,0.00,Jun-14,649.91,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36
3,4,10000,10000,10000.0,36 months,13.49,339.31,c,c1,10+ years,rent,49200.0,source_verified,Dec-11,fully_paid,0,other,20.00,0.0,Feb-96,1.0,35.0,9999.0,10.0,0.0,5598,21.0,37.0,fractional,0.0,0.0,12226.302210,12226.30,10000.00,2209.33,16.97,0.00,0.00,Jan-15,357.48,NaN,Jan-15,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,36
4,5,3000,3000,3000.0,60 months,12.69,67.79,b,b5,1 year,rent,80000.0,source_verified,Dec-11,current,0,other,17.94,0.0,Jan-96,0.0,38.0,9999.0,15.0,0.0,27783,53.9,38.0,fractional,766.9,766.9,3242.170000,3242.17,2233.10,1009.07,0.00,0.00,0.00,Jan-16,67.79,Feb-16,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,60


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months
0,1,35000,35000,35000.0,60 months,11.99,778.38,c,c1,10+ years,mortgage,128000.0,source_verified,Dec-15,issued,0,home_improvement,6.46,0.0,Feb-90,0.0,46.0,9999.0,17.0,0.0,14277,27.4,46.0,whole,35000.0,35000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1,60
1,2,8650,8650,8650.0,36 months,5.32,260.50,a,a1,< 1 year,mortgage,100000.0,not_verified,Dec-15,issued,0,credit_card,7.28,0.0,Jul-01,0.0,9999.0,9999.0,15.0,0.0,7158,26.7,24.0,whole,8650.0,8650.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0,36
2,3,4225,4225,4225.0,36 months,14.85,146.16,c,c5,5 years,rent,35000.0,source_verified,Dec-15,issued,0,debt_consolidation,15.22,2.0,Jul-11,0.0,18.0,9999.0,6.0,0.0,1058,24.6,6.0,whole,4225.0,4225.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0,36
3,4,10000,10000,10000.0,60 months,11.99,222.40,c,c1,10+ years,rent,42500.0,not_verified,Dec-15,issued,0,debt_consolidation,31.04,0.0,Dec-98,1.0,9999.0,9999.0,10.0,0.0,5812,40.9,23.0,whole,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0,60
4,5,20000,20000,20000.0,60 months,10.78,432.66,b,b4,10+ years,mortgage,63000.0,not_verified,Dec-15,issued,0,home_improvement,10.78,0.0,Aug-00,0.0,9999.0,9999.0,6.0,0.0,7869,56.2,18.0,whole,20000.0,20000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Dec-15,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0,60


row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                      string
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                           int64
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                      string
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                           int64
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

In [ ]:
# 6. emp_length ordinal
df_clean_training_data = transform_emp_length(df_clean_training_data, log_file=PROJECT_LOG_FILE)
df_clean_test_data = transform_emp_length(df_clean_test_data, log_file=PROJECT_LOG_FILE)

print(
    f"Train shape: {df_clean_training_data.shape} | "
    f"Test shape: {df_clean_test_data.shape}"
)

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_clean_training_data.head(5))
    display(df_clean_test_data.head(5))

display(df_clean_training_data.dtypes)
display(df_clean_test_data.dtypes)

Train shape: (466285, 53) | Test shape: (421094, 53)


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months,emp_length_years
0,1,5000,5000,4975.0,36 months,10.65,162.87,b,b2,10+ years,rent,24000.0,verified,Dec-11,fully_paid,0,credit_card,27.65,0.0,Jan-85,1.0,9999.0,9999.0,3.0,0.0,13648,83.7,9.0,fractional,0.0,0.0,5861.071414,5831.78,5000.00,861.07,0.00,0.00,0.00,Jan-15,171.62,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36,10.0
1,2,2500,2500,2500.0,60 months,15.27,59.83,c,c4,< 1 year,rent,30000.0,source_verified,Dec-11,charged_off,0,car,1.00,0.0,Apr-99,5.0,9999.0,9999.0,3.0,0.0,1687,9.4,4.0,fractional,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,Apr-13,119.66,NaN,Sep-13,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,60,0.0
2,3,2400,2400,2400.0,36 months,15.96,84.33,c,c5,10+ years,rent,12252.0,not_verified,Dec-11,fully_paid,0,small_business,8.72,0.0,Nov-01,2.0,9999.0,9999.0,2.0,0.0,2956,98.5,10.0,fractional,0.0,0.0,3003.653644,3003.65,2400.00,603.65,0.00,0.00,0.00,Jun-14,649.91,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36,10.0
3,4,10000,10000,10000.0,36 months,13.49,339.31,c,c1,10+ years,rent,49200.0,source_verified,Dec-11,fully_paid,0,other,20.00,0.0,Feb-96,1.0,35.0,9999.0,10.0,0.0,5598,21.0,37.0,fractional,0.0,0.0,12226.302210,12226.30,10000.00,2209.33,16.97,0.00,0.00,Jan-15,357.48,NaN,Jan-15,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,36,10.0
4,5,3000,3000,3000.0,60 months,12.69,67.79,b,b5,1 year,rent,80000.0,source_verified,Dec-11,current,0,other,17.94,0.0,Jan-96,0.0,38.0,9999.0,15.0,0.0,27783,53.9,38.0,fractional,766.9,766.9,3242.170000,3242.17,2233.10,1009.07,0.00,0.00,0.00,Jan-16,67.79,Feb-16,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,60,1.0


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months,emp_length_years
0,1,35000,35000,35000.0,60 months,11.99,778.38,c,c1,10+ years,mortgage,128000.0,source_verified,Dec-15,issued,0,home_improvement,6.46,0.0,Feb-90,0.0,46.0,9999.0,17.0,0.0,14277,27.4,46.0,whole,35000.0,35000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1,60,10.0
1,2,8650,8650,8650.0,36 months,5.32,260.50,a,a1,< 1 year,mortgage,100000.0,not_verified,Dec-15,issued,0,credit_card,7.28,0.0,Jul-01,0.0,9999.0,9999.0,15.0,0.0,7158,26.7,24.0,whole,8650.0,8650.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0,36,0.0
2,3,4225,4225,4225.0,36 months,14.85,146.16,c,c5,5 years,rent,35000.0,source_verified,Dec-15,issued,0,debt_consolidation,15.22,2.0,Jul-11,0.0,18.0,9999.0,6.0,0.0,1058,24.6,6.0,whole,4225.0,4225.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0,36,5.0
3,4,10000,10000,10000.0,60 months,11.99,222.40,c,c1,10+ years,rent,42500.0,not_verified,Dec-15,issued,0,debt_consolidation,31.04,0.0,Dec-98,1.0,9999.0,9999.0,10.0,0.0,5812,40.9,23.0,whole,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0,60,10.0
4,5,20000,20000,20000.0,60 months,10.78,432.66,b,b4,10+ years,mortgage,63000.0,not_verified,Dec-15,issued,0,home_improvement,10.78,0.0,Aug-00,0.0,9999.0,9999.0,6.0,0.0,7869,56.2,18.0,whole,20000.0,20000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Dec-15,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0,60,10.0


row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                      string
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                           int64
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                      string
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                           int64
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

In [ ]:
# 7. Month-year datetime conversions (including excluded/benchmark dates, per your choice)
df_clean_training_data = convert_month_year_columns_to_datetime(
    dataframe=df_clean_training_data,
    column_names=month_year_datetime_columns,
    log_file=PROJECT_LOG_FILE,
)
df_clean_test_data = convert_month_year_columns_to_datetime(
    dataframe=df_clean_test_data,
    column_names=month_year_datetime_columns,
    log_file=PROJECT_LOG_FILE,
)

print(
    f"Train shape: {df_clean_training_data.shape} | "
    f"Test shape: {df_clean_test_data.shape}"
)

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_clean_training_data.head(5))
    display(df_clean_test_data.head(5))

display(df_clean_training_data.dtypes)
display(df_clean_test_data.dtypes)

Train shape: (466285, 53) | Test shape: (421094, 53)


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months,emp_length_years
0,1,5000,5000,4975.0,36 months,10.65,162.87,b,b2,10+ years,rent,24000.0,verified,2011-12-01,fully_paid,0,credit_card,27.65,0.0,1985-01-01,1.0,9999.0,9999.0,3.0,0.0,13648,83.7,9.0,fractional,0.0,0.0,5861.071414,5831.78,5000.00,861.07,0.00,0.00,0.00,2015-01-01,171.62,NaT,2016-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36,10.0
1,2,2500,2500,2500.0,60 months,15.27,59.83,c,c4,< 1 year,rent,30000.0,source_verified,2011-12-01,charged_off,0,car,1.00,0.0,1999-04-01,5.0,9999.0,9999.0,3.0,0.0,1687,9.4,4.0,fractional,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,2013-04-01,119.66,NaT,2013-09-01,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,60,0.0
2,3,2400,2400,2400.0,36 months,15.96,84.33,c,c5,10+ years,rent,12252.0,not_verified,2011-12-01,fully_paid,0,small_business,8.72,0.0,2001-11-01,2.0,9999.0,9999.0,2.0,0.0,2956,98.5,10.0,fractional,0.0,0.0,3003.653644,3003.65,2400.00,603.65,0.00,0.00,0.00,2014-06-01,649.91,NaT,2016-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36,10.0
3,4,10000,10000,10000.0,36 months,13.49,339.31,c,c1,10+ years,rent,49200.0,source_verified,2011-12-01,fully_paid,0,other,20.00,0.0,1996-02-01,1.0,35.0,9999.0,10.0,0.0,5598,21.0,37.0,fractional,0.0,0.0,12226.302210,12226.30,10000.00,2209.33,16.97,0.00,0.00,2015-01-01,357.48,NaT,2015-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,36,10.0
4,5,3000,3000,3000.0,60 months,12.69,67.79,b,b5,1 year,rent,80000.0,source_verified,2011-12-01,current,0,other,17.94,0.0,1996-01-01,0.0,38.0,9999.0,15.0,0.0,27783,53.9,38.0,fractional,766.9,766.9,3242.170000,3242.17,2233.10,1009.07,0.00,0.00,0.00,2016-01-01,67.79,2016-02-01,2016-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,60,1.0


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months,emp_length_years
0,1,35000,35000,35000.0,60 months,11.99,778.38,c,c1,10+ years,mortgage,128000.0,source_verified,2015-12-01,issued,0,home_improvement,6.46,0.0,1990-02-01,0.0,46.0,9999.0,17.0,0.0,14277,27.4,46.0,whole,35000.0,35000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1,60,10.0
1,2,8650,8650,8650.0,36 months,5.32,260.50,a,a1,< 1 year,mortgage,100000.0,not_verified,2015-12-01,issued,0,credit_card,7.28,0.0,2001-07-01,0.0,9999.0,9999.0,15.0,0.0,7158,26.7,24.0,whole,8650.0,8650.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0,36,0.0
2,3,4225,4225,4225.0,36 months,14.85,146.16,c,c5,5 years,rent,35000.0,source_verified,2015-12-01,issued,0,debt_consolidation,15.22,2.0,2011-07-01,0.0,18.0,9999.0,6.0,0.0,1058,24.6,6.0,whole,4225.0,4225.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0,36,5.0
3,4,10000,10000,10000.0,60 months,11.99,222.40,c,c1,10+ years,rent,42500.0,not_verified,2015-12-01,issued,0,debt_consolidation,31.04,0.0,1998-12-01,1.0,9999.0,9999.0,10.0,0.0,5812,40.9,23.0,whole,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0,60,10.0
4,5,20000,20000,20000.0,60 months,10.78,432.66,b,b4,10+ years,mortgage,63000.0,not_verified,2015-12-01,issued,0,home_improvement,10.78,0.0,2000-08-01,0.0,9999.0,9999.0,6.0,0.0,7869,56.2,18.0,whole,20000.0,20000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2015-12-01,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0,60,10.0


row_id                                      int64
loan_amnt                                   int64
funded_amnt                                 int64
funded_amnt_inv                           float64
term                                          str
int_rate                                  float64
installment                               float64
grade                                         str
sub_grade                                     str
emp_length                                    str
home_ownership                             string
annual_inc                                float64
verification_status                           str
issue_d                            datetime64[ns]
loan_status                                   str
pymnt_plan                                  int64
purpose                                       str
dti                                       float64
delinq_2yrs                               float64
earliest_cr_line                   datetime64[ns]


row_id                                      int64
loan_amnt                                   int64
funded_amnt                                 int64
funded_amnt_inv                           float64
term                                          str
int_rate                                  float64
installment                               float64
grade                                         str
sub_grade                                     str
emp_length                                    str
home_ownership                             string
annual_inc                                float64
verification_status                           str
issue_d                            datetime64[ns]
loan_status                                   str
pymnt_plan                                  int64
purpose                                       str
dti                                       float64
delinq_2yrs                               float64
earliest_cr_line                   datetime64[ns]


In [ ]:
# 8. We drop old columns term and emp_legth
df_clean_training_data = drop_columns_with_logging(
    dataframe=df_clean_training_data,
    columns_to_drop=["term", "emp_length"],
    dataset_name="train_clean",
    log_file=PROJECT_LOG_FILE,
    errors='ignore'
)

df_clean_test_data = drop_columns_with_logging(
    dataframe=df_clean_test_data,
    columns_to_drop=["term", "emp_length"],
    dataset_name="test_clean",
    log_file=PROJECT_LOG_FILE,
    errors='ignore'
)

print(
    f"Train shape: {df_clean_training_data.shape} | "
    f"Test shape: {df_clean_test_data.shape}"
)

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_clean_training_data.head(5))
    display(df_clean_test_data.head(5))

display(df_clean_training_data.dtypes)
display(df_clean_test_data.dtypes)

Train shape: (466285, 51) | Test shape: (421094, 51)


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,grade,sub_grade,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months,emp_length_years
0,1,5000,5000,4975.0,10.65,162.87,b,b2,rent,24000.0,verified,2011-12-01,fully_paid,0,credit_card,27.65,0.0,1985-01-01,1.0,9999.0,9999.0,3.0,0.0,13648,83.7,9.0,fractional,0.0,0.0,5861.071414,5831.78,5000.00,861.07,0.00,0.00,0.00,2015-01-01,171.62,NaT,2016-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36,10.0
1,2,2500,2500,2500.0,15.27,59.83,c,c4,rent,30000.0,source_verified,2011-12-01,charged_off,0,car,1.00,0.0,1999-04-01,5.0,9999.0,9999.0,3.0,0.0,1687,9.4,4.0,fractional,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,2013-04-01,119.66,NaT,2013-09-01,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,60,0.0
2,3,2400,2400,2400.0,15.96,84.33,c,c5,rent,12252.0,not_verified,2011-12-01,fully_paid,0,small_business,8.72,0.0,2001-11-01,2.0,9999.0,9999.0,2.0,0.0,2956,98.5,10.0,fractional,0.0,0.0,3003.653644,3003.65,2400.00,603.65,0.00,0.00,0.00,2014-06-01,649.91,NaT,2016-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36,10.0
3,4,10000,10000,10000.0,13.49,339.31,c,c1,rent,49200.0,source_verified,2011-12-01,fully_paid,0,other,20.00,0.0,1996-02-01,1.0,35.0,9999.0,10.0,0.0,5598,21.0,37.0,fractional,0.0,0.0,12226.302210,12226.30,10000.00,2209.33,16.97,0.00,0.00,2015-01-01,357.48,NaT,2015-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,36,10.0
4,5,3000,3000,3000.0,12.69,67.79,b,b5,rent,80000.0,source_verified,2011-12-01,current,0,other,17.94,0.0,1996-01-01,0.0,38.0,9999.0,15.0,0.0,27783,53.9,38.0,fractional,766.9,766.9,3242.170000,3242.17,2233.10,1009.07,0.00,0.00,0.00,2016-01-01,67.79,2016-02-01,2016-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,60,1.0


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,grade,sub_grade,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months,emp_length_years
0,1,35000,35000,35000.0,11.99,778.38,c,c1,mortgage,128000.0,source_verified,2015-12-01,issued,0,home_improvement,6.46,0.0,1990-02-01,0.0,46.0,9999.0,17.0,0.0,14277,27.4,46.0,whole,35000.0,35000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1,60,10.0
1,2,8650,8650,8650.0,5.32,260.50,a,a1,mortgage,100000.0,not_verified,2015-12-01,issued,0,credit_card,7.28,0.0,2001-07-01,0.0,9999.0,9999.0,15.0,0.0,7158,26.7,24.0,whole,8650.0,8650.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0,36,0.0
2,3,4225,4225,4225.0,14.85,146.16,c,c5,rent,35000.0,source_verified,2015-12-01,issued,0,debt_consolidation,15.22,2.0,2011-07-01,0.0,18.0,9999.0,6.0,0.0,1058,24.6,6.0,whole,4225.0,4225.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0,36,5.0
3,4,10000,10000,10000.0,11.99,222.40,c,c1,rent,42500.0,not_verified,2015-12-01,issued,0,debt_consolidation,31.04,0.0,1998-12-01,1.0,9999.0,9999.0,10.0,0.0,5812,40.9,23.0,whole,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0,60,10.0
4,5,20000,20000,20000.0,10.78,432.66,b,b4,mortgage,63000.0,not_verified,2015-12-01,issued,0,home_improvement,10.78,0.0,2000-08-01,0.0,9999.0,9999.0,6.0,0.0,7869,56.2,18.0,whole,20000.0,20000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2015-12-01,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0,60,10.0


row_id                                      int64
loan_amnt                                   int64
funded_amnt                                 int64
funded_amnt_inv                           float64
int_rate                                  float64
installment                               float64
grade                                         str
sub_grade                                     str
home_ownership                             string
annual_inc                                float64
verification_status                           str
issue_d                            datetime64[ns]
loan_status                                   str
pymnt_plan                                  int64
purpose                                       str
dti                                       float64
delinq_2yrs                               float64
earliest_cr_line                   datetime64[ns]
inq_last_6mths                            float64
mths_since_last_delinq                    float64


row_id                                      int64
loan_amnt                                   int64
funded_amnt                                 int64
funded_amnt_inv                           float64
int_rate                                  float64
installment                               float64
grade                                         str
sub_grade                                     str
home_ownership                             string
annual_inc                                float64
verification_status                           str
issue_d                            datetime64[ns]
loan_status                                   str
pymnt_plan                                  int64
purpose                                       str
dti                                       float64
delinq_2yrs                               float64
earliest_cr_line                   datetime64[ns]
inq_last_6mths                            float64
mths_since_last_delinq                    float64


In [ ]:
# 9. Standardize dtypes for categorical columns
categorical_columns_to_cast = [
    "purpose",
    "home_ownership",
    "verification_status",
    "initial_list_status",
    "grade",
    "sub_grade",
    "loan_status",
]

try:
    log("Casting categorical columns: string -> category (train/test)")

    for column_name in categorical_columns_to_cast:
        if column_name not in df_clean_training_data.columns:
            log(f"Train missing categorical column (skipped): {column_name}")
            continue
        if column_name not in df_clean_test_data.columns:
            log(f"Test missing categorical column (skipped): {column_name}")
            continue

        # Convert both to pandas string first (keeps <NA> as missing)
        df_clean_training_data[column_name] = df_clean_training_data[column_name].astype("string")
        df_clean_test_data[column_name] = df_clean_test_data[column_name].astype("string")

        # Build a shared category space across train/test (prevents drift)
        train_values = df_clean_training_data[column_name].dropna().unique().tolist()
        test_values = df_clean_test_data[column_name].dropna().unique().tolist()
        combined_categories = sorted(set(train_values) | set(test_values))

        df_clean_training_data[column_name] = pd.Categorical(
            df_clean_training_data[column_name],
            categories=combined_categories
        )
        df_clean_test_data[column_name] = pd.Categorical(
            df_clean_test_data[column_name],
            categories=combined_categories
        )

        log(
            f"Casted '{column_name}' to category | "
            f"train_unique={len(train_values)} | test_unique={len(test_values)} | combined={len(combined_categories)}"
        )

except Exception as error:
    log(f"Error during categorical dtype casting: {error}")
    raise

print(f"Train shape: {df_clean_training_data.shape} | Test shape: {df_clean_test_data.shape}")

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_clean_training_data.head(5))
    display(df_clean_test_data.head(5))

display(df_clean_training_data.dtypes)
display(df_clean_test_data.dtypes)


Train shape: (466285, 51) | Test shape: (421094, 51)


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,grade,sub_grade,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months,emp_length_years
0,1,5000,5000,4975.0,10.65,162.87,b,b2,rent,24000.0,verified,2011-12-01,fully_paid,0,credit_card,27.65,0.0,1985-01-01,1.0,9999.0,9999.0,3.0,0.0,13648,83.7,9.0,fractional,0.0,0.0,5861.071414,5831.78,5000.00,861.07,0.00,0.00,0.00,2015-01-01,171.62,NaT,2016-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36,10.0
1,2,2500,2500,2500.0,15.27,59.83,c,c4,rent,30000.0,source_verified,2011-12-01,charged_off,0,car,1.00,0.0,1999-04-01,5.0,9999.0,9999.0,3.0,0.0,1687,9.4,4.0,fractional,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,2013-04-01,119.66,NaT,2013-09-01,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,60,0.0
2,3,2400,2400,2400.0,15.96,84.33,c,c5,rent,12252.0,not_verified,2011-12-01,fully_paid,0,small_business,8.72,0.0,2001-11-01,2.0,9999.0,9999.0,2.0,0.0,2956,98.5,10.0,fractional,0.0,0.0,3003.653644,3003.65,2400.00,603.65,0.00,0.00,0.00,2014-06-01,649.91,NaT,2016-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36,10.0
3,4,10000,10000,10000.0,13.49,339.31,c,c1,rent,49200.0,source_verified,2011-12-01,fully_paid,0,other,20.00,0.0,1996-02-01,1.0,35.0,9999.0,10.0,0.0,5598,21.0,37.0,fractional,0.0,0.0,12226.302210,12226.30,10000.00,2209.33,16.97,0.00,0.00,2015-01-01,357.48,NaT,2015-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,36,10.0
4,5,3000,3000,3000.0,12.69,67.79,b,b5,rent,80000.0,source_verified,2011-12-01,current,0,other,17.94,0.0,1996-01-01,0.0,38.0,9999.0,15.0,0.0,27783,53.9,38.0,fractional,766.9,766.9,3242.170000,3242.17,2233.10,1009.07,0.00,0.00,0.00,2016-01-01,67.79,2016-02-01,2016-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,60,1.0


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,grade,sub_grade,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months,emp_length_years
0,1,35000,35000,35000.0,11.99,778.38,c,c1,mortgage,128000.0,source_verified,2015-12-01,issued,0,home_improvement,6.46,0.0,1990-02-01,0.0,46.0,9999.0,17.0,0.0,14277,27.4,46.0,whole,35000.0,35000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1,60,10.0
1,2,8650,8650,8650.0,5.32,260.50,a,a1,mortgage,100000.0,not_verified,2015-12-01,issued,0,credit_card,7.28,0.0,2001-07-01,0.0,9999.0,9999.0,15.0,0.0,7158,26.7,24.0,whole,8650.0,8650.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0,36,0.0
2,3,4225,4225,4225.0,14.85,146.16,c,c5,rent,35000.0,source_verified,2015-12-01,issued,0,debt_consolidation,15.22,2.0,2011-07-01,0.0,18.0,9999.0,6.0,0.0,1058,24.6,6.0,whole,4225.0,4225.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0,36,5.0
3,4,10000,10000,10000.0,11.99,222.40,c,c1,rent,42500.0,not_verified,2015-12-01,issued,0,debt_consolidation,31.04,0.0,1998-12-01,1.0,9999.0,9999.0,10.0,0.0,5812,40.9,23.0,whole,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0,60,10.0
4,5,20000,20000,20000.0,10.78,432.66,b,b4,mortgage,63000.0,not_verified,2015-12-01,issued,0,home_improvement,10.78,0.0,2000-08-01,0.0,9999.0,9999.0,6.0,0.0,7869,56.2,18.0,whole,20000.0,20000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2015-12-01,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0,60,10.0


row_id                                      int64
loan_amnt                                   int64
funded_amnt                                 int64
funded_amnt_inv                           float64
int_rate                                  float64
installment                               float64
grade                                    category
sub_grade                                category
home_ownership                           category
annual_inc                                float64
verification_status                      category
issue_d                            datetime64[ns]
loan_status                              category
pymnt_plan                                  int64
purpose                                  category
dti                                       float64
delinq_2yrs                               float64
earliest_cr_line                   datetime64[ns]
inq_last_6mths                            float64
mths_since_last_delinq                    float64


row_id                                      int64
loan_amnt                                   int64
funded_amnt                                 int64
funded_amnt_inv                           float64
int_rate                                  float64
installment                               float64
grade                                    category
sub_grade                                category
home_ownership                           category
annual_inc                                float64
verification_status                      category
issue_d                            datetime64[ns]
loan_status                              category
pymnt_plan                                  int64
purpose                                  category
dti                                       float64
delinq_2yrs                               float64
earliest_cr_line                   datetime64[ns]
inq_last_6mths                            float64
mths_since_last_delinq                    float64


In [ ]:
# Check columns with NaN values that are now categorical after transformations.
columns_to_check = [
    "tot_coll_amt",
    "tot_cur_bal",
    "total_rev_hi_lim",
]

# Training null report
training_null_report = (
    df_clean_training_data[columns_to_check]
    .isna()
    .agg(["sum", "mean"])
    .T
)

training_null_report["mean"] = (training_null_report["mean"] * 100).round(2)
training_null_report.columns = ["null_count_train", "null_percent_train"]

# Test null report
test_null_report = (
    df_clean_test_data[columns_to_check]
    .isna()
    .agg(["sum", "mean"])
    .T
)

test_null_report["mean"] = (test_null_report["mean"] * 100).round(2)
test_null_report.columns = ["null_count_test", "null_percent_test"]

null_comparison = training_null_report.join(test_null_report)

null_comparison


,null_count_train,null_percent_train,null_count_test,null_percent_test
tot_coll_amt,70276.0,15.07,0.0,0.0
tot_cur_bal,70276.0,15.07,0.0,0.0
total_rev_hi_lim,70276.0,15.07,0.0,0.0


In [ ]:
# Drop additional columns from the clean training dataset to create the modeling datasets
df_feature_base_training_data = drop_columns_with_logging(
    dataframe=df_clean_training_data,
    columns_to_drop=excluded_columns + benchmark_columns,
    dataset_name="train_feature_base",
    log_file=PROJECT_LOG_FILE,
    errors='raise'
    )

df_feature_base_test_data = drop_columns_with_logging(
    dataframe=df_clean_test_data,
    columns_to_drop=excluded_columns + benchmark_columns,
    dataset_name="test_feature_base",
    log_file=PROJECT_LOG_FILE,
    errors='raise'
    )

print(
    f"Train shape: {df_feature_base_training_data.shape} | "
    f"Test shape: {df_feature_base_test_data.shape}"
)

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_feature_base_training_data.head(5))
    display(df_feature_base_test_data.head(5))

display(df_feature_base_training_data.dtypes)
display(df_feature_base_test_data.dtypes)

Train shape: (466285, 27) | Test shape: (421094, 27)


,row_id,loan_amnt,home_ownership,annual_inc,loan_status,purpose,dti,delinq_2yrs,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months,emp_length_years
0,1,5000,rent,24000.0,fully_paid,credit_card,27.65,0.0,1.0,9999.0,9999.0,3.0,0.0,13648,83.7,9.0,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36,10.0
1,2,2500,rent,30000.0,charged_off,car,1.00,0.0,5.0,9999.0,9999.0,3.0,0.0,1687,9.4,4.0,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,60,0.0
2,3,2400,rent,12252.0,fully_paid,small_business,8.72,0.0,2.0,9999.0,9999.0,2.0,0.0,2956,98.5,10.0,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36,10.0
3,4,10000,rent,49200.0,fully_paid,other,20.00,0.0,1.0,35.0,9999.0,10.0,0.0,5598,21.0,37.0,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,36,10.0
4,5,3000,rent,80000.0,current,other,17.94,0.0,0.0,38.0,9999.0,15.0,0.0,27783,53.9,38.0,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,60,1.0


,row_id,loan_amnt,home_ownership,annual_inc,loan_status,purpose,dti,delinq_2yrs,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months,emp_length_years
0,1,35000,mortgage,128000.0,issued,home_improvement,6.46,0.0,0.0,46.0,9999.0,17.0,0.0,14277,27.4,46.0,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1,60,10.0
1,2,8650,mortgage,100000.0,issued,credit_card,7.28,0.0,0.0,9999.0,9999.0,15.0,0.0,7158,26.7,24.0,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0,36,0.0
2,3,4225,rent,35000.0,issued,debt_consolidation,15.22,2.0,0.0,18.0,9999.0,6.0,0.0,1058,24.6,6.0,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0,36,5.0
3,4,10000,rent,42500.0,issued,debt_consolidation,31.04,0.0,1.0,9999.0,9999.0,10.0,0.0,5812,40.9,23.0,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0,60,10.0
4,5,20000,mortgage,63000.0,issued,home_improvement,10.78,0.0,0.0,9999.0,9999.0,6.0,0.0,7869,56.2,18.0,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0,60,10.0


row_id                                int64
loan_amnt                             int64
home_ownership                     category
annual_inc                          float64
loan_status                        category
purpose                            category
dti                                 float64
delinq_2yrs                         float64
inq_last_6mths                      float64
mths_since_last_delinq              float64
mths_since_last_record              float64
open_acc                            float64
pub_rec                             float64
revol_bal                             int64
revol_util                          float64
total_acc                           float64
collections_12_mths_ex_med          float64
mths_since_last_major_derog         float64
acc_now_delinq                      float64
tot_coll_amt                        float64
tot_cur_bal                         float64
total_rev_hi_lim                    float64
has_mths_since_last_delinq      

row_id                                int64
loan_amnt                             int64
home_ownership                     category
annual_inc                          float64
loan_status                        category
purpose                            category
dti                                 float64
delinq_2yrs                         float64
inq_last_6mths                      float64
mths_since_last_delinq              float64
mths_since_last_record              float64
open_acc                            float64
pub_rec                             float64
revol_bal                             int64
revol_util                          float64
total_acc                           float64
collections_12_mths_ex_med          float64
mths_since_last_major_derog         float64
acc_now_delinq                      float64
tot_coll_amt                        float64
tot_cur_bal                         float64
total_rev_hi_lim                    float64
has_mths_since_last_delinq      

In [ ]:
# Basic sanity checks on the feature base training dataset before we save them
assert "loan_status" in df_feature_base_training_data.columns
set(df_clean_training_data.columns) - set(df_feature_base_training_data.columns)


{'collection_recovery_fee',
 'earliest_cr_line',
 'funded_amnt',
 'funded_amnt_inv',
 'grade',
 'initial_list_status',
 'installment',
 'int_rate',
 'issue_d',
 'last_credit_pull_d',
 'last_pymnt_amnt',
 'last_pymnt_d',
 'next_pymnt_d',
 'out_prncp',
 'out_prncp_inv',
 'pymnt_plan',
 'recoveries',
 'sub_grade',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_int',
 'total_rec_late_fee',
 'total_rec_prncp',
 'verification_status'}

In [ ]:
# Basic sanity checks on the feature base testdataset before we save them
assert "loan_status" in df_feature_base_test_data.columns
set(df_clean_test_data.columns) - set(df_feature_base_test_data.columns)

{'collection_recovery_fee',
 'earliest_cr_line',
 'funded_amnt',
 'funded_amnt_inv',
 'grade',
 'initial_list_status',
 'installment',
 'int_rate',
 'issue_d',
 'last_credit_pull_d',
 'last_pymnt_amnt',
 'last_pymnt_d',
 'next_pymnt_d',
 'out_prncp',
 'out_prncp_inv',
 'pymnt_plan',
 'recoveries',
 'sub_grade',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_int',
 'total_rec_late_fee',
 'total_rec_prncp',
 'verification_status'}

In [ ]:
# Feature-base schema must match exactly (order + names) before modeling
train_columns = df_feature_base_training_data.columns.tolist()
test_columns = df_feature_base_test_data.columns.tolist()

assert train_columns == test_columns, (
    "Train/test feature_base columns are not identical.\n"
    f"Only in train: {sorted(set(train_columns) - set(test_columns))}\n"
    f"Only in test: {sorted(set(test_columns) - set(train_columns))}"
)


In [ ]:
dtype_mismatches = []
for column_name in train_columns:
    if df_feature_base_training_data[column_name].dtype != df_feature_base_test_data[column_name].dtype:
        dtype_mismatches.append(
            (column_name,
             str(df_feature_base_training_data[column_name].dtype),
             str(df_feature_base_test_data[column_name].dtype))
        )

assert not dtype_mismatches, f"Dtype mismatches found: {dtype_mismatches[:20]}"


In [ ]:
# Saving all datasets to interim storage as parquet files
log = get_logger(PROJECT_LOG_FILE)

log(
    f"Final shapes | "
    f"train_clean={df_clean_training_data.shape} | "
    f"test_clean={df_clean_test_data.shape} | "
    f"train_feature_base={df_feature_base_training_data.shape} | "
    f"test_feature_base={df_feature_base_test_data.shape}"
)


log(f"Saving parquet | dataset=train_clean | path={clean_training_data_file}")
df_clean_training_data.to_parquet(clean_training_data_file, index=False)

log(f"Saving parquet | dataset=test_clean | path={clean_test_data_file}")
df_clean_test_data.to_parquet(clean_test_data_file, index=False)

log(f"Saving parquet | dataset=train_feature_base | path={feature_base_training_data_file}")
df_feature_base_training_data.to_parquet(feature_base_training_data_file, index=False)

log(f"Saving parquet | dataset=test_feature_base | path={feature_base_test_data_file}")
df_feature_base_test_data.to_parquet(feature_base_test_data_file, index=False)

# Notebook Conclusion – Data Cleaning & Feature Base Construction

## Objective

This notebook established the structural conditions required for modeling under a fixed constraint: prediction occurs at application submission.

All transformation decisions were derived from that constraint and from observed properties of the 2007–2014 training data.

---

## Structural Boundary Enforcement

The following exclusions were applied:

- All identifiers and export artifacts.
- All fully null variables in the training window.
- All zero-variance variables.
- All underwriting, pricing, funding, and servicing variables.
- Geographic proxies (`addr_state`, `zip_code`).

Variables absent in the 2007–2014 training window were removed from both training and test partitions. A feature that cannot be learned in training cannot be introduced at prediction time.

Result: The remaining feature space contains only borrower-declared and bureau-reported information observable at submission.

---

## Categorical and Type Normalization

Object-based fields were normalized and typed explicitly:

- Text standardized for consistent categorical representation.
- `emp_length` mapped to an ordinal integer scale (0–10), preserving missing values.
- Binary indicators encoded deterministically.
- Categorical variables cast to `category` dtype.
- Object-encoded numerics converted to numeric dtype.
- Month–year strings converted to `datetime64[ns]`.

No frequency filtering, target encoding, or feature engineering was introduced.

Result: Train and test partitions share identical categorical representation and datatypes.

---

## Vintage-Based Reporting Differences

The variables:

- `tot_coll_amt`
- `tot_cur_bal`
- `total_rev_hi_lim`

contain ~15% missing values in the 2007–2014 training window and no missing values in 2015.

This reflects historical reporting differences rather than transformation error.

Imputation strategy will be fit exclusively on the training window during modeling to preserve temporal integrity.

---

## Dataset Construction

Two dataset layers were created:

- `clean` — structurally normalized dataset retaining benchmark variables.
- `feature_base` — submission-time eligible variables plus the target.

For both training and test partitions:

- Column presence verified.
- Column order aligned.
- Datatypes matched.
- Schema checks executed.

No mismatches remain.

---

## Final State

The `feature_base` dataset contains only variables observable at application submission and the target.

No underwriting-derived, pricing-derived, or post-origination variables remain.

No feature present in the test partition is absent in training.

Categorical spaces are aligned across partitions.

Datetime fields are explicitly typed.

Geographic proxies have been removed.

Any predictive signal extracted from this dataset must arise from borrower-declared or bureau-reported information observable at submission.

---

## Next Phase

The next notebook will:

- Define the modeling cohort from `loan_status`.
- Perform EDA under explicit temporal awareness.
- Evaluate feature stability and predictive contribution.
- Implement train-fitted imputation and encoding pipelines.